# Scraping

### import library

In [32]:
_= !pip install PyPDF2
from bs4 import BeautifulSoup, Tag
import requests
import PyPDF2
import re
import pandas as pd
import zipfile
import os

### scraping direktory page

In [33]:
def scraping_putusan(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        putusan_hukum = soup.find_all('div', class_='spost clearfix')

        putusan_text = []
        putusan_filename = []
        putusan_meta = []

        # Mendapatkan setiap keputusan dari halaman
        for i in putusan_hukum:
            tag = i.find_all('strong')
            for y in tag:
                if y.find('a'):
                    result = get_data(y.find('a').get('href'))
                    if result:
                        [pdf_name, get_data_text, meta_info] = result
                        if meta_info:
                            putusan_text.append(get_data_text)
                            putusan_filename.append(pdf_name)
                            putusan_meta.append(meta_info)

        # Memeriksa jika ada halaman berikutnya
        buttons = soup.find_all('ul', class_='pagination justify-content-center')
        for button in buttons:
            tag_a = button.find_all('a')
            for link in tag_a:
                if link.text.strip() == 'Next':
                    return (putusan_filename, putusan_text, putusan_meta, link.get('href'))

    else:
        print("Gagal mengakses halaman web. Kode status:", response.status_code)

### scraping decision page

In [34]:
def get_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        box_lampiran = soup.find_all('ul', class_='portfolio-meta nobottommargin')[1]

        # Cek apakah ada tautan PDF di dalam lampiran
        if box_lampiran.find_all('a'):
            link_pdf = box_lampiran.find_all('a')[1]
            response_pdf = requests.get(link_pdf.get('href'))

            # Mengunduh file PDF jika tautan tersedia
            if response_pdf.status_code == 200:
                pdf_name = link_pdf.text.strip().replace('/', '_')
                with open(pdf_name, 'wb') as pdf_file:
                    pdf_file.write(response_pdf.content)
                print(f"File PDF '{pdf_name}' berhasil diunduh.")

                meta_info = generate_meta_info(soup, url)
                if meta_info:
                    meta_info.append(link_pdf.get('href'))
                print(f"Meta Info '{pdf_name}' berhasil dibuat.")
                return pdf_name, read_pdf(pdf_name), meta_info

            else:
                print("Gagal mengunduh file PDF. Kode status:", response.status_code)
                return False
        else:
            print('Link Download Tidak Tersedia')
            return False

### get metadata from decision page

In [35]:
def generate_meta_info(soup, url):
    # Daftar elemen metadata yang akan diekstraksi
    listMetaHead = ["url_page", "penuntut_umum", "terdakwa", "nomor", "kata_kunci", "tahun", "tanggal_register",
                    "lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar", "amar_lainnya",
                    "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan"]

    listMeta = [""] * len(listMetaHead)
    listMeta[0] = url

    # Mencari tabel metadata dalam halaman
    rowsMETA = soup.find("ul", {"class": "portfolio-meta nobottommargin"}).find("table").findAll("tr")
    for i, row in enumerate(rowsMETA):
        coll = row.findAll("td")
        if len(coll) > 1:
            # Menyesuaikan nama kolom untuk mencocokkan dengan listMetaHead
            formatted_coll_0 = clean_html_tags(str(coll[0])).lower().replace(" ", "_")
            index_in_listMetaHead = next((index for index, item in enumerate(listMetaHead) if item.lower() == formatted_coll_0), None)

            if index_in_listMetaHead is not None:
                cleantext = clean_html_tags(str(coll[1]))
                listMeta[index_in_listMetaHead] = cleantext
        else:
            h2_element = coll[0].find('h2')
            if h2_element and isinstance(h2_element, Tag):
                # Mengolah teks di dalam elemen H2 untuk mendapatkan informasi Penuntut Umum dan Terdakwa
                text_h = re.sub(r'<\/?br\/?>|<\/?strong\/?>', ' ', ' '.join(str(i) for i in h2_element))
                text_h = re.sub(r'\s+', ' ', text_h).strip()
                print(url)
                data = re.search(r'Penuntut Umum: (.+?) Terdakwa: (.+?)?(?=$|\s—)', text_h)
                if data:
                    Penuntun_Umum = data.group(1)
                    Terdakwa = data.group(2)

                    listMeta[1] = Penuntun_Umum
                    listMeta[2] = Terdakwa

    return listMeta

### cleaning text from html tags and unused conversion results

In [36]:
def clean_text(text):
    # Menghapus teks header dan disclaimer yang berulang dari dokumen
    text = text.replace("Mahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nDirektori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n", "")
    text = text.replace("\nDisclaimer\nKepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\npelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\nDalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\nEmail : kepaniteraan@mahkamahagung.go.id", "")
    text = text.replace("Telp : 021-384 3348 (ext.318)", "")
    # Menghapus nomor halaman dan elemen unik
    text = re.sub(r'\nHalaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'Halaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'\nHal. \d+ dari \d+ .*', '', text)
    text = re.sub(r'Hal. \d+ dari \d+ .*', '', text)
    text = re.sub(r' +|[\uf0fc\uf0a7\uf0a8\uf0b7]', ' ', text) # Menghapus bullet points dan spasi berlebih
    text = re.sub(r'[\u2026]+|\.{3,}', '', text) # Menghapus elipsis atau titik berulang
    return text.strip() # Menghapus spasi di awal dan akhir

In [37]:
def clean_html_tags(text):
    # Menghapus semua tag HTML dari teks
    clean_tags = re.compile('<.*?>')
    return re.sub(clean_tags, '', text).strip()

### save documents and convert to text

In [38]:
def read_pdf(file_pdf):
    try:
        pdf_text = ''
        pdf_file = open(file_pdf, 'rb')
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        # Membaca setiap halaman dan membersihkan teks
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = clean_text(page.extract_text())
            pdf_text += ' ' + text

        pdf_file.close()
        return pdf_text.strip() # Mengembalikan teks bersih

    except requests.exceptions.RequestException as e:
        print("Error:", e)

In [39]:
def zip_dokumen(filename, listData):
    # Membuat file zip dan menambahkan setiap dokumen
    with zipfile.ZipFile(filename, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file in listData:
            file_path = os.path.join('/content', file)
            zipf.write(file_path, os.path.basename(file_path))

    print(f"File {filename} berhasil diciptakan.")

### scraping implementation

In [40]:
# PATH = '/content/drive/MyDrive/Skripsi/Code/Data/'
# url = 'https://putusan3.mahkamahagung.go.id/search.html?q=&jenis_doc=putusan&cat=6f5068aaa5515f46e4aeafca2f48dfa2&jd=&tp=&court=098413PN310+++++++++++++++++++++&t_put=&t_reg=&t_upl=&t_pr=' #bangkalan
# filename = f"data_bkl.csv"

# putusan_file = []
# putusan_text = []
# putusan_meta = []
# next_button = url

# for page in range(1,2):
#   print(f"============== Page {page} ==============")
#   result = scraping_putusan(next_button,filename)
#   [putusan_file_new, putusan_text_new, putusan_meta_new, next_button_new] = result
#   putusan_file += putusan_file_new
#   putusan_text += putusan_text_new
#   putusan_meta += putusan_meta_new
#   next_button = next_button_new

# print(f'Berhasil mengunduh sebanyak {len(putusan_text)} File')

# # Menyimpan Datset Dalam Bentuk CSV
# df_meta = pd.DataFrame(putusan_meta, columns=["url_page", "penuntut_umum", "terdakwa", "nomor", "kata_kunci", "tahun", "tanggal_register",
#                   "lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar", "amar_lainnya",
#                   "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan", "url_pdf"])
# df_file = pd.DataFrame(putusan_file, columns=['nama_file'])
# df_text = pd.DataFrame(putusan_text, columns=['content'])
# df_concat = pd.concat([df_meta, df_file, df_text], axis=1)
# df_concat.to_csv(f'{PATH}{filename}', index=False)

# print(f'Berhasil menyimpan ke dalam file {filename}')
# zip_dokumen(f'{PATH}dokumen_bkl.zip', putusan_file)


In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
# data_bkl = pd.read_csv(f"{PATH}/data_pn_bkl.csv")
# data_spg = pd.read_csv(f"{PATH}/data_pn_spg.csv")
# data_pmk = pd.read_csv(f"{PATH}/data_pn_pmk.csv")
# data_smp = pd.read_csv(f"{PATH}/data_pn_smp.csv")
# data = pd.concat([data_bkl, data_spg, data_pmk, data_smp], ignore_index=True)
# data.to_csv(f"{PATH}/data_all.csv", index=False)
data = pd.read_csv(f"{PATH}/data_all.csv")
data

,url_page,penuntut_umum,terdakwa,nomor,kata_kunci,tahun,tanggal_register,lembaga_peradilan,hakim_ketua,hakim_anggota,panitera,amar,amar_lainnya,catatan_amar,tanggal_musyawarah,tanggal_dibacakan,url_pdf,nama_file,content
0,https://putusan3.mahkamahagung.go.id/direktori...,"DEWI IKA AGUSTINA, SH",TISLAN bin MUKAMAR,48/Pid.B/2024/PN Bkl,Pencurian,2024,15 Maret 2024,PN BANGKALAN,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: Menyatakan Terdakwa TISLAN BIN MUKA...,2 Mei 2024,2 Mei 2024,https://putusan3.mahkamahagung.go.id/direktori...,48_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 48/Pid.B/2024/PN Bkl\nDEMI KEAD...
1,https://putusan3.mahkamahagung.go.id/direktori...,"NANDINI PARAHITA YULISANI, S.H.",BEIRI Bin BUNALI,42/Pid.B/2024/PN Bkl,Penganiayaan,2024,5 Maret 2024,PN BANGKALAN,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Rr. Koosbandriyah As,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: Menyatakan Terdakwa Beiri bin Bunal...,2 Mei 2024,2 Mei 2024,https://putusan3.mahkamahagung.go.id/direktori...,42_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 42/Pid.B/2024/PN Bkl\nDEMI KEAD...
2,https://putusan3.mahkamahagung.go.id/direktori...,"UMU LATHIEFAH, S.H",FANDI bin SULAIMAN,49/Pid.B/2024/PN Bkl,Pencurian,2024,15 Maret 2024,PN BANGKALAN,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: Menyatakan Terdakwa FANDI BIN SULAI...,2 Mei 2024,2 Mei 2024,https://putusan3.mahkamahagung.go.id/direktori...,49_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 49/Pid.B/2024/PN Bkl\nDEMI KEAD...
3,https://putusan3.mahkamahagung.go.id/direktori...,"FAJRINI FAISAH, S.H.",PANDI Bin SIDIK,56/Pid.B/2024/PN Bkl,PenadahanPenerbitandan Pencetakan,2024,15 Maret 2024,PN BANGKALAN,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Naruddin,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: MenyatakanTerdakwaPandi Bin Sidikte...,2 Mei 2024,2 Mei 2024,https://putusan3.mahkamahagung.go.id/direktori...,56_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 56/Pid.B/2024/PN Bkl\nDEMI KEAD...
4,https://putusan3.mahkamahagung.go.id/direktori...,"FAJRINI FAISAH, S.H.",MUSTOFA al. JARWOT bin HADENAN (Alm),57/Pid.B/2024/PN Bkl,Pencurian,2024,15 Maret 2024,PN BANGKALAN,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: H. Mohammad Asari,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,M E N G A D I L I Menyatakan Terdakwa Mustofa ...,2 Mei 2024,2 Mei 2024,https://putusan3.mahkamahagung.go.id/direktori...,57_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 57/Pid.B/2024/PN Bkl\nDEMI KEAD...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
744,https://putusan3.mahkamahagung.go.id/direktori...,"NUR FAJJRIYAH, SH",ZAIFUL ISLAM Bin MUDIRUN,274/Pid.Sus/2021/PN Smp,Perlindungan Anak,2022,10 Nopember 2021,PN SUMENEP,Hakim Ketua Yahya Wahyudi,"Br Hakim Anggota Anjar Kumboro, Hakim Anggota...",Panitera Pengganti Edi Haris Mulyono,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: 1. Menyatakan Terdakwa ZAIFUL ...,20 Januari 2022,20 Januari 2022,https://putusan3.mahkamahagung.go.id/direktori...,274_Pid.Sus_2021_PN_Smp.pdf,PUTUSAN \nNomor 274/Pid.Sus/2021/PN Smp \n \nD...
745,https://putusan3.mahkamahagung.go.id/direktori...,HARRY ACHMAD DWI MARYONO,ABDUL MUGHNIS Bin JAUHARI,272/Pid.B/2021/PN Smp,Penganiayaan,2022,10 Nopember 2021,PN SUMENEP,Hakim Ketua Yahya Wahyudi,"Hakim Anggota Muhammad Arief Fatony, Br Hakim...",Panitera Pengganti Sugiarto,Lain-lain,PIDANA PENJARA WAKTU TERTENTU,MENGADILI: Menyatakan Terdakwa Abdul Mughnis B...,20 Januari 2022,20 Januari 2022,https://putusan3.mahkamahagung.go.id/direktori...,272_Pid.B_2021_PN_Smp.pdf,PUTUSAN \nNomor 272/Pid.B/2021/PN Smp \n \nDEM...
746,https://putusan3.mahkamahagung.go.

### create an excel to store an entity

In [43]:
# data['tindak_pidana'] = ''
# data['pelanggaran_kuhp'] = ''
# data['putusan_hukuman'] = ''
# data['tuntutan_hukuman'] = ''
# data[['nama_file', 'penuntut_umum', 'terdakwa', 'nomor', 'tanggal_musyawarah', 'hakim_ketua',
#        'hakim_anggota', 'panitera', 'tindak_pidana', 'tuntutan_hukuman', 'pelanggaran_kuhp',
#        'putusan_hukuman']].to_excel("entity_data_all.xlsx", index=False)

In [44]:
entities = pd.read_excel(f"{PATH}/entity_data_all.xlsx")
entities

,nama_file,penuntut_umum,terdakwa,nomor,tanggal_musyawarah,hakim_ketua,hakim_anggota,panitera,tindak_pidana,tuntutan_hukuman,pelanggaran_kuhp,putusan_hukuman
0,48_Pid.B_2024_PN_Bkl,"DEWI IKA AGUSTINA, SH",TISLAN bin MUKAMAR,48/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (dua) tahun 2 (dua) bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
1,42_Pid.B_2024_PN_Bkl,"NANDINI PARAHITA YULISANI, S.H.",BEIRI Bin BUNALI,42/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Rr. Koosbandriyah As,Secara Melawan Hukum Memaksa Orang Lain Supaya...,6 (enam) bulan,Pasal 335 ayat (1) Kitab Undang-Undang Hukum P...,5 (lima) bulan
2,49_Pid.B_2024_PN_Bkl,"UMU LATHIEFAH, S.H",FANDI bin SULAIMAN,49/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (Dua) Tahun 2 (Dua) Bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
3,56_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",PANDI Bin SIDIK,56/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Naruddin,penadahan,6 (enam) bulan,Pasal 480 ke-1 Kitab Undang-Undang Hukum Pidana,4 (empat) Bulan 15 (lima belas) Hari
4,57_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",MUSTOFA al. JARWOT bin HADENAN (Alm),57/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: H. Mohammad Asari,pencurian dalam keadaan memberatkan,2 (dua) tahun dan 5 (lima) bulan,Pasal 363 ayat (1) ke-3 dan ke-4 KUHP,2 (dua) tahum
...,...,...,...,...,...,...,...,...,...,...,...,...
744,274_Pid.Sus_2021_PN_Smp,"NUR FAJJRIYAH, SH",ZAIFUL ISLAM Bin MUDIRUN,274/Pid.Sus/2021/PN Smp,20 Januari 2022,Hakim Ketua Yahya Wahyudi,"Br Hakim Anggota Anjar Kumboro, Hakim Anggota...",Panitera Pengganti Edi Haris Mulyono,dengan sengaja membujuk anak melakukan Persetu...,7 (Tujuh) Tahun,Pasal 81 ayat (2) Undang - Undang Republik Ind...,"5 (lima ) Tahun,"
745,272_Pid.B_2021_PN_Smp,HARRY ACHMAD DWI MARYONO,ABDUL MUGHNIS Bin JAUHARI,272/Pid.B/2021/PN Smp,20 Januari 2022,Hakim Ketua Yahya Wahyudi,"Hakim Anggota Muhammad Arief Fatony, Br Hakim...",Panitera Pengganti Sugiarto,Penganiayaan,5 ( LIMA ) Bulan,Pasal 351 ayat (1) Kitab Undang -Undang Hukum ...,3 (tiga) bulan dan 5 (lima) hari
746,220_Pid.Sus_2021_PN_Smp,"NUR FAJJRIYAH, SH",SUKRAWI Bin SUPATMO,220/Pid.Sus/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Sugiarto,Dengan sengaja melakukan tipu muslihat memaksa...,9 ( sembilan ) Tahun,Pasal 82 ayat (2) U ndang - Undang Republik In...,6 (enam) tahun
747,293_Pid.B_2021_PN_Smp,"R. TEDDY ROOMIUS, SH",1.ASNAWI BIN SYAFI I 2.HORMAH BIN SAMLI,293/Pid.B/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Nani Irianingsih,Perjudian,4 (empat ) bulan,Pasal 303 Ayat (l) ke.2 KUHP,3 (tiga) bulan


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Preprocessing

In [46]:
from tqdm import tqdm
import pandas as pd
import gdown
import re

load dataset


In [47]:
PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'

In [48]:
data = pd.read_csv(f"{PATH}/data_all.csv")
data = data[['nama_file', 'content']]
data

,nama_file,content
0,48_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 48/Pid.B/2024/PN Bkl\nDEMI KEAD...
1,42_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 42/Pid.B/2024/PN Bkl\nDEMI KEAD...
2,49_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 49/Pid.B/2024/PN Bkl\nDEMI KEAD...
3,56_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 56/Pid.B/2024/PN Bkl\nDEMI KEAD...
4,57_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 57/Pid.B/2024/PN Bkl\nDEMI KEAD...
...,...,...
744,274_Pid.Sus_2021_PN_Smp.pdf,PUTUSAN \nNomor 274/Pid.Sus/2021/PN Smp \n \nD...
745,272_Pid.B_2021_PN_Smp.pdf,PUTUSAN \nNomor 272/Pid.B/2021/PN Smp \n \nDEM...
746,220_Pid.Sus_2021_PN_Smp.pdf,PUTUSAN \nNomor 220/Pid.Sus/2021/PN Smp \n \nD...
747,293_Pid.B_2021_PN_Smp.pdf,PUTUSAN \nNomor 293/Pid.B/2021/PN Smp \n \nDEM...


In [49]:
data['content'][0]

'PUTUSAN\nNomor 48/Pid.B/2024/PN Bkl\nDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA\nPengadilan Negeri Bangkalan yang mengadili perkara pidana dengan\nacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai\nberikut dalam perkara Terdakwa :\n1. Nama lengkap : Tislan Bin Mukamar\n2. Tempat lahir : Bangkalan\n3. Umur/Tanggal lahir : 35 Tahun/12 April 1989\n4. Jenis kelamin : Laki-laki\n5. Kebangsaan : Indonesia\n6. Tempat tinggal : Dsn. Lang Lebar Ds. Batobella Kec. Geger Kab. \n Bangkalan\n7. Agama : Islam\n8. Pekerjaan : Wiraswasta\nTerdakwa Tislan Bin Mukamar ditahan dalam tahanan oleh: \n1. Penyidik sejak tanggal 15 Januari 2024 sampai dengan tanggal 3 Februari\n2024 \n2. Penyidik Perpanjangan Oleh Penuntut Umum sejak tanggal 4 Februari 2024\nsampai dengan tanggal 14 Maret 2024 \n3. Penuntut Umum sejak tanggal 13 Maret 2024 sampai dengan tanggal 1 April\n2024 \n4. Hakim Pengadilan Negeri sejak tanggal 15 Maret 2024 sampai dengan\ntanggal 13 April 2024 \n5. Hakim

In [50]:
entities = pd.read_excel(f"{PATH}/entities_all.xlsx")
entities

,nama_file,penuntut_umum,terdakwa,nomor,tanggal_musyawarah,hakim_ketua,hakim_anggota,panitera,tindak_pidana,tuntutan_hukuman,pelanggaran_kuhp,putusan_hukuman
0,48_Pid.B_2024_PN_Bkl,"DEWI IKA AGUSTINA, SH",TISLAN bin MUKAMAR,48/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (dua) tahun 2 (dua) bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
1,42_Pid.B_2024_PN_Bkl,"NANDINI PARAHITA YULISANI, S.H.",BEIRI Bin BUNALI,42/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Rr. Koosbandriyah A .S,Secara Melawan Hukum Memaksa Orang Lain Supaya...,6 (enam) bulan,Pasal 335 ayat (1) Kitab Undang-Undang Hukum P...,5 (lima) bulan
2,49_Pid.B_2024_PN_Bkl,"UMU LATHIEFAH, S.H",FANDI bin SULAIMAN,49/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (Dua) Tahun 2 (Dua) Bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
3,56_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",PANDI Bin SIDIK,56/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Naruddin,penadahan,6 (enam) bulan,Pasal 480 ke-1 Kitab Undang-Undang Hukum Pidana,4 (empat) Bulan 15 (lima belas) Hari
4,57_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",MUSTOFA al. JARWOT bin HADENAN (Alm),57/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: H. Mohammad Asari,pencurian dalam keadaan memberatkan,2 (dua) tahun dan 5 (lima) bulan,Pasal 363 ayat (1) ke-3 dan ke-4 KUHP,2 (dua) tahum
...,...,...,...,...,...,...,...,...,...,...,...,...
744,274_Pid.Sus_2021_PN_Smp,"Nur Fajjriy ah, S.H",ZAIFUL ISLAM Bin MUDIRUN,274/Pid.Sus/2021/PN Smp,20 Januari 2022,Hakim Ketua Yahya Wahyudi,"Br Hakim Anggota Anjar Kumboro, Hakim Anggota...",Panitera Pengganti Edi Haris Mulyono,dengan sengaja membujuk anak melakukan Persetu...,7 (Tujuh) Tahun,Pasal 81 ayat (2) Undang - Undang Republik Ind...,"5 (LIMA) Tahun,"
745,272_Pid.B_2021_PN_Smp,HARRY ACHMAD DWI MARYONO,ABDUL MUGHNIS Bin JAUHARI,272/Pid.B/2021/PN Smp,13 Januari 2022,Hakim Ketua Yahya Wahyudi,"Hakim Anggota Muhammad Arief Fatony, Br Hakim...",Panitera Pengganti Sugiarto,Penganiayaan,5 (lima) Bulan,Pasal 351 ayat (1) Kitab Undang -Undang Hukum ...,3 (tiga) bulan dan 5 (lima) hari
746,220_Pid.Sus_2021_PN_Smp,"NUR FAJJRIYAH, SH",SUKRAWI Bin SUPATMO,220/Pid.Sus/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Sugiarto,Dengan sengaja melakukan tipu muslihat memaksa...,9 (sembilan) Tahun,Pasal 82 ayat (2) U ndang - Undang Republik In...,6 (enam) tahun
747,293_Pid.B_2021_PN_Smp,"R.Tedy Roomius ,S.H.",1.ASNAWI BIN SYAFI I 2.HORMAH BIN SAMLI,293/Pid.B/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Nani Irianingsih,Perjudian,4 (empat) bulan,Pasal 303 Ayat (l) ke.2 KUHP,3 (tiga) bulan


In [51]:
entities.columns

Index(['nama_file', 'penuntut_umum', 'terdakwa', 'nomor', 'tanggal_musyawarah',
       'hakim_ketua', 'hakim_anggota', 'panitera', 'tindak_pidana',
       'tuntutan_hukuman', 'pelanggaran_kuhp', 'putusan_hukuman'],
      dtype='object')

In [52]:
missing_values = entities.isnull().sum()
missing_values

,0
nama_file,0
penuntut_umum,123
terdakwa,1
nomor,0
tanggal_musyawarah,0
hakim_ketua,0
hakim_anggota,0
panitera,0
tindak_pidana,6
tuntutan_hukuman,13


hapus data yang terdapat missing value

In [53]:
entities = entities.dropna()
entities = entities.applymap(str)
entities

/tmp/ipykernel_696/3809814227.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  entities = entities.applymap(str)


,nama_file,penuntut_umum,terdakwa,nomor,tanggal_musyawarah,hakim_ketua,hakim_anggota,panitera,tindak_pidana,tuntutan_hukuman,pelanggaran_kuhp,putusan_hukuman
0,48_Pid.B_2024_PN_Bkl,"DEWI IKA AGUSTINA, SH",TISLAN bin MUKAMAR,48/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (dua) tahun 2 (dua) bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
1,42_Pid.B_2024_PN_Bkl,"NANDINI PARAHITA YULISANI, S.H.",BEIRI Bin BUNALI,42/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Rr. Koosbandriyah A .S,Secara Melawan Hukum Memaksa Orang Lain Supaya...,6 (enam) bulan,Pasal 335 ayat (1) Kitab Undang-Undang Hukum P...,5 (lima) bulan
2,49_Pid.B_2024_PN_Bkl,"UMU LATHIEFAH, S.H",FANDI bin SULAIMAN,49/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: Hosnol Bakri,Pencurian Dalam Keadaan Memberatkan,2 (Dua) Tahun 2 (Dua) Bulan,Pasal 363 ayat (2) KUHP,1 (satu) Tahun dan 10 (sepuluh) bulan
3,56_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",PANDI Bin SIDIK,56/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Zainal Ahmad,"Br Hakim Anggota Wienda Kresnantyo, Hakim Ang...",Panitera Pengganti: Naruddin,penadahan,6 (enam) bulan,Pasal 480 ke-1 Kitab Undang-Undang Hukum Pidana,4 (empat) Bulan 15 (lima belas) Hari
4,57_Pid.B_2024_PN_Bkl,"FAJRINI FAISAH, S.H.",MUSTOFA al. JARWOT bin HADENAN (Alm),57/Pid.B/2024/PN Bkl,2 Mei 2024,Hakim Ketua Ernila Widikartikawati,"Br Hakim Anggota Wahyu Eko Suryowati, Hakim A...",Panitera Pengganti: H. Mohammad Asari,pencurian dalam keadaan memberatkan,2 (dua) tahun dan 5 (lima) bulan,Pasal 363 ayat (1) ke-3 dan ke-4 KUHP,2 (dua) tahum
...,...,...,...,...,...,...,...,...,...,...,...,...
744,274_Pid.Sus_2021_PN_Smp,"Nur Fajjriy ah, S.H",ZAIFUL ISLAM Bin MUDIRUN,274/Pid.Sus/2021/PN Smp,20 Januari 2022,Hakim Ketua Yahya Wahyudi,"Br Hakim Anggota Anjar Kumboro, Hakim Anggota...",Panitera Pengganti Edi Haris Mulyono,dengan sengaja membujuk anak melakukan Persetu...,7 (Tujuh) Tahun,Pasal 81 ayat (2) Undang - Undang Republik Ind...,"5 (LIMA) Tahun,"
745,272_Pid.B_2021_PN_Smp,HARRY ACHMAD DWI MARYONO,ABDUL MUGHNIS Bin JAUHARI,272/Pid.B/2021/PN Smp,13 Januari 2022,Hakim Ketua Yahya Wahyudi,"Hakim Anggota Muhammad Arief Fatony, Br Hakim...",Panitera Pengganti Sugiarto,Penganiayaan,5 (lima) Bulan,Pasal 351 ayat (1) Kitab Undang -Undang Hukum ...,3 (tiga) bulan dan 5 (lima) hari
746,220_Pid.Sus_2021_PN_Smp,"NUR FAJJRIYAH, SH",SUKRAWI Bin SUPATMO,220/Pid.Sus/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Sugiarto,Dengan sengaja melakukan tipu muslihat memaksa...,9 (sembilan) Tahun,Pasal 82 ayat (2) U ndang - Undang Republik In...,6 (enam) tahun
747,293_Pid.B_2021_PN_Smp,"R.Tedy Roomius ,S.H.",1.ASNAWI BIN SYAFI I 2.HORMAH BIN SAMLI,293/Pid.B/2021/PN Smp,4 Januari 2022,Hakim Ketua Yuniar Yudha Himawan,"Hakim Anggota Yahya Wahyudi, Br Hakim Anggota...",Panitera Pengganti Nani Irianingsih,Perjudian,4 (empat) bulan,Pasal 303 Ayat (l) ke.2 KUHP,3 (tiga) bulan


In [54]:
data = data.loc[entities.index]
data

,nama_file,content
0,48_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 48/Pid.B/2024/PN Bkl\nDEMI KEAD...
1,42_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 42/Pid.B/2024/PN Bkl\nDEMI KEAD...
2,49_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 49/Pid.B/2024/PN Bkl\nDEMI KEAD...
3,56_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 56/Pid.B/2024/PN Bkl\nDEMI KEAD...
4,57_Pid.B_2024_PN_Bkl.pdf,PUTUSAN\nNomor 57/Pid.B/2024/PN Bkl\nDEMI KEAD...
...,...,...
744,274_Pid.Sus_2021_PN_Smp.pdf,PUTUSAN \nNomor 274/Pid.Sus/2021/PN Smp \n \nD...
745,272_Pid.B_2021_PN_Smp.pdf,PUTUSAN \nNomor 272/Pid.B/2021/PN Smp \n \nDEM...
746,220_Pid.Sus_2021_PN_Smp.pdf,PUTUSAN \nNomor 220/Pid.Sus/2021/PN Smp \n \nD...
747,293_Pid.B_2021_PN_Smp.pdf,PUTUSAN \nNomor 293/Pid.B/2021/PN Smp \n \nDEM...


## formatting data

In [55]:
def format_penuntut_umum(penuntut_umum):
    penuntut_umum = re.sub(r'\b(S\.H\.?|M\.H\.?|SH|MH)\b', '', penuntut_umum).replace(',', '')
    if any(char.isdigit() for char in penuntut_umum):
        penuntut_umum = re.findall(r'\d+\.(.*?)\s*(?=\d+\.|$)', penuntut_umum.lower())
    else:
        penuntut_umum = [penuntut_umum.lower().strip()]

    penuntut_umum = [name.rstrip('.') for name in penuntut_umum]
    penuntut_umum = [re.sub(r'\s*\.\s*$', '', name).strip() for name in penuntut_umum]
    return penuntut_umum

def format_terdakwa(terdakwa):
    if any(char.isdigit() for char in terdakwa):
        terdakwa = re.findall(r'\d+\.(.*?)\s*(?=\d+\.|$)', terdakwa.lower())
    else:
        terdakwa = [terdakwa.lower().strip()]
    return terdakwa

def format_tuntutan(tuntutan):
    tuntutan_list = tuntutan.lower().split(';')
    tuntutan_list = [tuntutan.strip() for tuntutan in tuntutan_list]
    return tuntutan_list

def format_putusan(putusan):
    putusan_list = putusan.lower().split(';')
    putusan_list = [putusan.strip() for putusan in putusan_list]
    return putusan_list

entities.fillna('', inplace=True)
entities["penuntut_umum"] = entities["penuntut_umum"].apply(lambda x: format_penuntut_umum(x))
entities["terdakwa"] = entities["terdakwa"].apply(lambda x: format_terdakwa(x))
entities["nomor"] = entities["nomor"].apply(lambda x: x.lower().strip())
entities["tanggal_musyawarah"] = entities["tanggal_musyawarah"].apply(lambda x: re.sub(r'\b0+(\d+)', r'\1', x.lower().replace('nopember', 'november').strip()))
entities["hakim_ketua"] = entities["hakim_ketua"].apply(lambda x: x.lower().replace("hakim ketua", '').replace("hakim tunggal", '').strip())
entities["hakim_anggota"] = entities["hakim_anggota"].apply(lambda x: re.sub(r'\bbr\b', '', x.lower().replace("hakim anggota", '').replace("hakim tunggal", '')).strip().split(','))
entities["panitera"] = entities["panitera"].apply(lambda x: x.lower().replace("panitera pengganti", "").replace(":", "").strip())
entities["tindak_pidana"] = entities["tindak_pidana"].apply(lambda x: str(x).lower().strip() if isinstance(x, str) else x)
entities["tuntutan_hukuman"] = entities["tuntutan_hukuman"].apply(lambda x: format_tuntutan(str(x)) if isinstance(x, str) else x)
entities["pelanggaran_kuhp"] = entities["pelanggaran_kuhp"].apply(lambda x: str(x).lower().strip() if isinstance(x, str) else x)
entities["putusan_hukuman"] = entities["putusan_hukuman"].apply(lambda x: format_putusan(str(x)) if isinstance(x, str) else x)
entities

,nama_file,penuntut_umum,terdakwa,nomor,tanggal_musyawarah,hakim_ketua,hakim_anggota,panitera,tindak_pidana,tuntutan_hukuman,pelanggaran_kuhp,putusan_hukuman
0,48_Pid.B_2024_PN_Bkl,[dewi ika agustina],[tislan bin mukamar],48/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",hosnol bakri,pencurian dalam keadaan memberatkan,[2 (dua) tahun 2 (dua) bulan],pasal 363 ayat (2) kuhp,[1 (satu) tahun dan 10 (sepuluh) bulan]
1,42_Pid.B_2024_PN_Bkl,[nandini parahita yulisani],[beiri bin bunali],42/pid.b/2024/pn bkl,2 mei 2024,zainal ahmad,"[wienda kresnantyo, kadek dwi krisna ananda]",rr. koosbandriyah a .s,secara melawan hukum memaksa orang lain supaya...,[6 (enam) bulan],pasal 335 ayat (1) kitab undang-undang hukum p...,[5 (lima) bulan]
2,49_Pid.B_2024_PN_Bkl,[umu lathiefah],[fandi bin sulaiman],49/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",hosnol bakri,pencurian dalam keadaan memberatkan,[2 (dua) tahun 2 (dua) bulan],pasal 363 ayat (2) kuhp,[1 (satu) tahun dan 10 (sepuluh) bulan]
3,56_Pid.B_2024_PN_Bkl,[fajrini faisah],[pandi bin sidik],56/pid.b/2024/pn bkl,2 mei 2024,zainal ahmad,"[wienda kresnantyo, kadek dwi krisna ananda]",naruddin,penadahan,[6 (enam) bulan],pasal 480 ke-1 kitab undang-undang hukum pidana,[4 (empat) bulan 15 (lima belas) hari]
4,57_Pid.B_2024_PN_Bkl,[fajrini faisah],[mustofa al. jarwot bin hadenan (alm)],57/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",h. mohammad asari,pencurian dalam keadaan memberatkan,[2 (dua) tahun dan 5 (lima) bulan],pasal 363 ayat (1) ke-3 dan ke-4 kuhp,[2 (dua) tahum]
...,...,...,...,...,...,...,...,...,...,...,...,...
744,274_Pid.Sus_2021_PN_Smp,[nur fajjriy ah],[zaiful islam bin mudirun],274/pid.sus/2021/pn smp,20 januari 2022,yahya wahyudi,"[anjar kumboro, muhammad arief fatony]",edi haris mulyono,dengan sengaja membujuk anak melakukan persetu...,[7 (tujuh) tahun],pasal 81 ayat (2) undang - undang republik ind...,"[5 (lima) tahun,]"
745,272_Pid.B_2021_PN_Smp,[harry achmad dwi maryono],[abdul mughnis bin jauhari],272/pid.b/2021/pn smp,13 januari 2022,yahya wahyudi,"[muhammad arief fatony, anjar kumboro]",sugiarto,penganiayaan,[5 (lima) bulan],pasal 351 ayat (1) kitab undang -undang hukum ...,[3 (tiga) bulan dan 5 (lima) hari]
746,220_Pid.Sus_2021_PN_Smp,[nur fajjriyah],[sukrawi bin supatmo],220/pid.sus/2021/pn smp,4 januari 2022,yuniar yudha himawan,"[yahya wahyudi, anjar kumboro]",sugiarto,dengan sengaja melakukan tipu muslihat memaksa...,[9 (sembilan) tahun],pasal 82 ayat (2) u ndang - undang republik in...,[6 (enam) tahun]
747,293_Pid.B_2021_PN_Smp,[r.tedy roomius],"[asnawi bin syafi i, hormah bin samli]",293/pid.b/2021/pn smp,4 januari 2022,yuniar yudha himawan,"[yahya wahyudi, anjar kumboro]",nani irianingsih,perjudian,[4 (empat) bulan],pasal 303 ayat (l) ke.2 kuhp,[3 (tiga) bulan]


## cleaning text v1

In [56]:
def clean_text(text):
    text = text.replace('P U T U S A N', 'PUTUSAN').replace('T erdakwa', 'Terdakwa').replace('T empat', 'Tempat').replace('T ahun', 'Tahun')
    text = text.replace('P  E  N  E  T  A  P  A  N', 'PENETAPAN').replace('J u m l a h', 'Jumlah').replace('M E N G A D I L I', 'MENGADILI')
    text = re.sub(r'Halaman \d+', ' ', text)
    text = re.sub(r'\b0+(\d+)', r'\1', text)
    text = text.replace('\uf0d8', '').replace('\uf0b7', '').replace('\n', ' ')
    text = re.sub(r'([“”"])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

data['content'] = data["content"].apply(clean_text)

## clean text v2

In [57]:
# def clean_text(text):
#     # Ganti kata yang dipisahkan dengan spasi menjadi bentuk yang benar
#     text = text.replace('P U T U S A N', 'PUTUSAN').replace('T erdakwa', 'Terdakwa').replace('T empat', 'Tempat').replace('T ahun', 'Tahun')
#     text = text.replace('P  E  N  E  T  A  P  A  N', 'PENETAPAN').replace('J u m l a h', 'Jumlah').replace('M E N G A D I L I', 'MENGADILI')
#     # Hapus pola tertentu seperti "Hal.", "Halaman", dan "Putusan Nomor"
#     text = re.sub(r'(Hal\.\s*\S+(?:\s*\S+)?\.?\s*)|(Halaman \d+(?:\.\d+)?\s*)|Putusan Nomor \S+\s*', '', text)
#     # Hapus karakter Unicode yang tidak diinginkan dan ganti newline dengan spasi
#     text = text.replace('\uf0d8', '').replace('\uf0b7', '').replace('\n', ' ')
#     # Tambahkan spasi di sekitar tanda kutip
#     text = re.sub(r'([“”"])', r' \1 ', text)
#     # Ganti beberapa spasi dengan satu spasi
#     text = re.sub(r'\s+', ' ', text)
#     # Hapus pola "halaman x dari y"
#     text = re.sub(r'halaman\s*\d+\s*dari\s*\d+\s*', '', text)
#     # Hapus pola "dari x bkl" dan "dari x smp" di awal teks, tidak sensitif huruf besar/kecil
#     text = re.sub(r'^\s*dari\s+\d+\s+bkl\s+', '', text, flags=re.IGNORECASE)
#     text = re.sub(r'^\s*dari\s+\d+\s+smp\s+', '', text, flags=re.IGNORECASE)
#     # Perbaiki spasi sebelum pola "/PN"
#     text = re.sub(r'\s+/PN', '/PN', text)
#     # Tambahkan spasi antara "nomor" dan angka berikutnya
#     text = re.sub(r'(?i)(nomor)(\d+)', r'\1 \2', text)
#     # Hapus spasi di dalam tanda kurung
#     text = re.sub(r'\(\s*(.*?)\s*\)', r'(\1)', text)
#     # Tambahkan spasi setelah tanda titik dua jika tidak ada
#     text = re.sub(r':\s*(\S)', r': \1', text)
#     # Tambahkan spasi sebelum "SH" dan "S" setelah koma, tidak sensitif huruf besar/kecil
#     text = re.sub(r',\s*(sh)', r' ,\1', text, flags=re.IGNORECASE)
#     text = re.sub(r',\s*(s)', r' ,\1', text, flags=re.IGNORECASE)
#     # Hapus spasi antara angka dan teks di dalam tanda kurung
#     text = re.sub(r'(\d)\s*\((.*?)\)', r'\1 (\2)', text)
#     # Perbaiki spasi sebelum pola "/pid"
#     text = re.sub(r'(\d+)\s*/pid', r'\1/pid', text)
#     # Standarisasi format untuk nomor kasus
#     text = re.sub(r'(\d+)\s*/\s*(pid\.\w+)\s*/\s*(\d{4})\s*/\s*(pn)', r'\1/\2/\3/\4', text, flags=re.IGNORECASE)

#     return text.strip()

# def multiple_replace(text, replacements):
#     regex = re.compile('|'.join(re.escape(key) for key in replacements.keys()))

#     def replace(match):
#         return replacements[match.group(0)]

#     return regex.sub(replace, text)

# replacements = {
#     '1 0 (sepuluh)': '10 (sepuluh)', 'Nopember': 'November','Pebruari': 'Februari', 'Halaman1dari13Putusan' : 'Putusan', ':ABDUL' : 'ABDUL', 'A STUTI' :'ASTUTI', "SA'AT" : 'SA’AT', "Rifa'i" : 'Rifa’i',
#     'N omor': 'Nomor', 'K itab': 'Kitab', 'Hakmelakukan': 'Hak melakukan', '20 20': '2020', 'ke manfatan': 'kemanfatan', '2 7' : '27', 'A gustus' : 'Agustus', 'September2023,' : 'September 2023,',
#     '( 2)': '(2)', '.,': ' .,', ',S': ' ,S', 'PN.Pmk': 'PN Pmk', 'Pencuriandalam': 'Pencurian dalam', 'B in' : 'bin', 'Aapril' : 'April', 'Ja nuari' : 'Januari', 'Ted dy' : 'Teddy', 'Ro omius,' : 'Roomius,',
#     'Pembunuhanberencana': 'Pembunuhan berencana', 'h ukum': 'hukum', 'PNPmk': 'PN Pmk', '//PN': '/PN','R .B.': 'R. B.', '2022oleh' : '2022', 'olehFajrini' : 'oleh Fajrini', 'Alimuddi n,' : 'Alimuddin,',
#     '202 3/PN': '2023/PN', 'Perkos aan': 'Perkosaan', 'memberatka n': 'memberatkan', 'p idana': 'pidana', 'sebag aimana': 'sebagaimana', '1 0' : '10','bulan;' : 'bulan', '2 23' : '2023',
#     'Ram dhani': 'Ramdhani', 'H ADY': 'HADY', '2 023': '2023','2 022': '2022', 'f isik': 'fisik', 'be rsalah': 'bersalah', '(sepuluh )': '(sepuluh)', 'M U A R I' : 'MUARI', 'Bulan;' : 'Bulan', 'Ap ril' : 'April',
#     '(enam )': '(enam)', 'No mor': 'Nomor', '( 3)': '(3)', 'Nomo r': 'Nomor', 'an ak': 'anak', 'tuj uh': 'tujuh', 'hu bungan': 'hubungan', 'KADIR;' : 'KADIR', 'selam a1 (satu)' : 'selama 1 (satu)',
#     '(empat )' : '(empat)', "PA'I" : 'PA’I', 'S eptember' : 'September',
#     'Keadaa n' : 'Keadaan', '27Januari' : '27 Januari', 'Nomor265/Pid.B/2021/PN' : 'Nomor 265/Pid.B/2021/PN', 'Bulan;3.Menetapkan' : 'Bulan 3.Menetapkan','bulan;3.Menetapkan' : 'bulan 3.Menetapkan', 'Januari,2023' : 'Januari 2023',
#     '202 2' : '2022','202 4' : '2024', '202 3' : '2023','2023//' : '2023', '20 22' : '2022','20 21' : '2021','202 1' : '2021', '27januari' : '27 januari', 'januari,2023' : 'januari 2023', 'sepember' : 'september',
#     'Bk l' : 'Bkl', 'bk l' : 'bkl','pm k' : 'pmk', 'bki' : 'bkl',  'pid.b /' : 'pid.b/', 'pn.' : 'pn ', 'pid. b' : 'pid.b', 'pnpmk' : 'pn pmk', '2023pn' : '2023 pn', 'smp.' : 'smp',
#     '(al m)' : '(alm)', 'Al m' : 'Alm', "'" : '’', 'b in' : 'bin', 'bu lan' : 'bulan', 'kuhpdan' : 'kuhp dan',
#     'pencuriandalam' : 'pencurian dalam', 'keadaa n' : 'keadaan', 'pembunuhanberencana' : 'pembunuhan berencana', 'bulan3.menetapkan' : 'bulan 3.menetapkan', 'bulan3.menyatakan' :  'bulan 3.menyatakan',
#     't aufik' : 'taufik', 'olehhaidir' : 'oleh haidir','olehfajrini' : 'oleh fajrini', 'janurai' : 'januari', 'nopember': 'november','pebruari': 'februari','a gustus' : 'agustus', 'september2023,' : 'september 2023,','aapril' : 'april', 'ja nuari' : 'januari',
#     'no mor': 'nomor', ',serta' : ', serta', 'agussyamsul' : 'agus syamsul', 'a stuti' : 'astuti'
# }

# data['content'] = data["content"].apply(clean_text).apply(lambda x: multiple_replace(x, replacements))
# data

## anotation

In [58]:
data_concat = pd.concat([entities, data['content']], axis=1)
data_concat.reset_index(drop=True, inplace=True)
data_concat

,nama_file,penuntut_umum,terdakwa,nomor,tanggal_musyawarah,hakim_ketua,hakim_anggota,panitera,tindak_pidana,tuntutan_hukuman,pelanggaran_kuhp,putusan_hukuman,content
0,48_Pid.B_2024_PN_Bkl,[dewi ika agustina],[tislan bin mukamar],48/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",hosnol bakri,pencurian dalam keadaan memberatkan,[2 (dua) tahun 2 (dua) bulan],pasal 363 ayat (2) kuhp,[1 (satu) tahun dan 10 (sepuluh) bulan],PUTUSAN Nomor 48/Pid.B/2024/PN Bkl DEMI KEADIL...
1,42_Pid.B_2024_PN_Bkl,[nandini parahita yulisani],[beiri bin bunali],42/pid.b/2024/pn bkl,2 mei 2024,zainal ahmad,"[wienda kresnantyo, kadek dwi krisna ananda]",rr. koosbandriyah a .s,secara melawan hukum memaksa orang lain supaya...,[6 (enam) bulan],pasal 335 ayat (1) kitab undang-undang hukum p...,[5 (lima) bulan],PUTUSAN Nomor 42/Pid.B/2024/PN Bkl DEMI KEADIL...
2,49_Pid.B_2024_PN_Bkl,[umu lathiefah],[fandi bin sulaiman],49/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",hosnol bakri,pencurian dalam keadaan memberatkan,[2 (dua) tahun 2 (dua) bulan],pasal 363 ayat (2) kuhp,[1 (satu) tahun dan 10 (sepuluh) bulan],PUTUSAN Nomor 49/Pid.B/2024/PN Bkl DEMI KEADIL...
3,56_Pid.B_2024_PN_Bkl,[fajrini faisah],[pandi bin sidik],56/pid.b/2024/pn bkl,2 mei 2024,zainal ahmad,"[wienda kresnantyo, kadek dwi krisna ananda]",naruddin,penadahan,[6 (enam) bulan],pasal 480 ke-1 kitab undang-undang hukum pidana,[4 (empat) bulan 15 (lima belas) hari],PUTUSAN Nomor 56/Pid.B/2024/PN Bkl DEMI KEADIL...
4,57_Pid.B_2024_PN_Bkl,[fajrini faisah],[mustofa al. jarwot bin hadenan (alm)],57/pid.b/2024/pn bkl,2 mei 2024,ernila widikartikawati,"[wahyu eko suryowati, satrio budiono]",h. mohammad asari,pencurian dalam keadaan memberatkan,[2 (dua) tahun dan 5 (lima) bulan],pasal 363 ayat (1) ke-3 dan ke-4 kuhp,[2 (dua) tahum],PUTUSAN Nomor 57/Pid.B/2024/PN Bkl DEMI KEADIL...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
597,274_Pid.Sus_2021_PN_Smp,[nur fajjriy ah],[zaiful islam bin mudirun],274/pid.sus/2021/pn smp,20 januari 2022,yahya wahyudi,"[anjar kumboro, muhammad arief fatony]",edi haris mulyono,dengan sengaja membujuk anak melakukan persetu...,[7 (tujuh) tahun],pasal 81 ayat (2) undang - undang republik ind...,"[5 (lima) tahun,]",PUTUSAN Nomor 274/Pid.Sus/2021/PN Smp DEMI KEA...
598,272_Pid.B_2021_PN_Smp,[harry achmad dwi maryono],[abdul mughnis bin jauhari],272/pid.b/2021/pn smp,13 januari 2022,yahya wahyudi,"[muhammad arief fatony, anjar kumboro]",sugiarto,penganiayaan,[5 (lima) bulan],pasal 351 ayat (1) kitab undang -undang hukum ...,[3 (tiga) bulan dan 5 (lima) hari],PUTUSAN Nomor 272/Pid.B/2021/PN Smp DEMI KEADI...
599,220_Pid.Sus_2021_PN_Smp,[nur fajjriyah],[sukrawi bin supatmo],220/pid.sus/2021/pn smp,4 januari 2022,yuniar yudha himawan,"[yahya wahyudi, anjar kumboro]",sugiarto,dengan sengaja melakukan tipu muslihat memaksa...,[9 (sembilan) tahun],pasal 82 ayat (2) u ndang - undang republik in...,[6 (enam) tahun],PUTUSAN Nomor 220/Pid.Sus/2021/PN Smp DEMI KEA...
600,293_Pid.B_2021_PN_Smp,[r.tedy roomius],"[asnawi bin syafi i, hormah bin samli]",293/pid.b/2021/pn smp,4 januari 2022,yuniar yudha himawan,"[yahya wahyudi, anjar kumboro]",nani irianingsih,perjudian,[4 (empat) bulan],pasal 303 ayat (l) ke.2 kuhp,[3 (tiga) bulan],PUTUSAN Nomor 293/Pid.B/2021/PN Smp DEMI KEADI...


### create label dictionary

In [59]:
def generate_label(entities):
  label = {}

  def update_label(entity_list, label, label_value):
    if entity_list:  # Periksa apakah entity_list tidak kosong
        if len(entity_list) > 1:
            label.update({ent.strip(): label_value for ent in entity_list})
        else:
            if (entity_list[0]) == '':
              pass
            else:
              label[entity_list[0].strip()] = label_value

  # Periksa dan perbarui kamus label untuk setiap entitas
  if 'penuntut_umum' in entities and entities['penuntut_umum']:
      update_label(entities['penuntut_umum'], label, "B_PROS")

  if 'terdakwa' in entities and entities['terdakwa']:
      update_label(entities['terdakwa'], label, "B_DEFN")

  if 'hakim_anggota' in entities and entities['hakim_anggota']:
      update_label(entities['hakim_anggota'], label, "B_JUDG")

  if 'putusan_hukuman' in entities and entities['putusan_hukuman']:
      update_label(entities['putusan_hukuman'], label, "B_PUNI")

  if 'tuntutan_hukuman' in entities and entities['tuntutan_hukuman']:
      update_label(entities['tuntutan_hukuman'], label, "B_PENA")

  label[entities['nomor']] = "B_VERN"
  label[entities['tanggal_musyawarah']] = "B_TIMV"
  label[entities['hakim_ketua']] = "B_JUDP"
  label[entities['panitera']] = "B_REGI"
  label[entities['tindak_pidana']] = "B_CRIA"
  label[entities['pelanggaran_kuhp']] = "B_ARTV"

  return label

### otomatis anotation

In [60]:
def labeling_token(text, label_dict):
    tokens = text.lower().split()
    labeled_tokens = ['O'] * len(tokens)
    find_label = dict()

    # Buat struktur nested berdasarkan label_dict
    for key, value in label_dict.items():
        if key:  # Pastikan kunci tidak kosong
            words = key.split()
            current_dict = find_label
            for word in words[:-1]:
                word = re.sub(r'[.,;"]', '', word)  # Hapus tanda baca
                current_dict = current_dict.setdefault(word, {})
            if words:
                last_word = re.sub(r'[.,;"]', '', words[-1])
                current_dict[last_word] = {'label': value}

    # List untuk menyimpan indeks token sementara
    temp_idx_token = []
    # Struktur label yang sedang dicari
    find_label_selected = find_label
    # Iterasi melalui token
    for i, token in enumerate(tokens):
        clean_token = re.sub(r'[.,;"]', '', token)  # Hapus tanda baca
        if clean_token in find_label_selected:
            find_label_selected = find_label_selected[clean_token]
            temp_idx_token.append(i)
            if 'label' in find_label_selected:
                # Temukan label untuk token saat ini dan tetapkan ke token
                tag = find_label_selected['label']
                new_label = [tag if ind == 0 else tag.replace('B', 'I') for ind in range(len(temp_idx_token))]
                labeled_tokens[temp_idx_token[0]:temp_idx_token[0] + len(new_label)] = new_label
                temp_idx_token = []
                find_label_selected = find_label
        else:
            temp_idx_token = []
            find_label_selected = find_label

    return labeled_tokens

In [61]:
def anotasi(data):
    content_results = pd.DataFrame(columns=["doc", "text", "label"])
    count_label = {}
    labels = []

    for doc_idx, (_, row) in tqdm(enumerate(data.iterrows()), total=len(data), desc="Annotating"):
        temp_results = {"doc": [], "text": [], "label": []}

        label = generate_label(row)
        labels.append(label)
        content = row["content"]
        labeled_tokens = labeling_token(content, label)

        doc_str = f"doc:{doc_idx+1}"
        temp_results["doc"].append(doc_str)
        temp_results["text"].append(content.split())
        temp_results["label"].append(labeled_tokens)
        temp_results['label'] = handleDuplicatePena(temp_results['label'])

        temp_data = pd.DataFrame(temp_results)
        count_label.update(countLabel(f"{doc_idx}", temp_data, set(label.values())))
        content_results = pd.concat([content_results, temp_data], ignore_index=True)

    return content_results, count_label, labels

In [62]:
def handleDuplicatePena(sequences):
    countPena = 0

    for sequence in sequences:
        for i, label in enumerate(sequence):
            if label == 'B_PENA':
                countPena += 1
                if countPena > 1:
                    sequence[i] = 'B_PUNI'
            elif label == 'I_PENA':
                if countPena > 1:
                    sequence[i] = 'I_PUNI'

    return sequences

In [63]:
def countLabel(doc_idx, data, label_ori):
    result = {}
    label_found = set()
    label_list = set(['B_ARTV', 'B_DEFN', 'B_PROS', 'B_CRIA', 'B_JUDP', 'B_VERN', 'B_PUNI', 'B_JUDG', 'B_REGI', 'B_TIMV', 'B_PENA'])
    for i, row in data.iterrows():
      for t, l in zip(row["text"], row["label"]):
        if l != 'O' and l.replace('I_', 'B_') not in label_found:
          label_found.add(l.replace('I_', 'B_'))

    label_missing = label_list - label_found
    result[f"Dokumen Ke-{doc_idx}"] = (len(label_found), len(label_ori), label_missing)
    return result

def accuracy(result):
  sum_label = 0
  label_in = 0
  for key, value in result.items():
    print(f"{key} => Data Terlebel : {value[0]} ; Total Label : {value[1]} ; Label Hilang : {value[2]}")
    sum_label+= value[1]
    label_in += value[0]

  return f"Accuracy : {label_in/sum_label * 100}"

new_df, count_label, labels = anotasi(data_concat)
new_df

Annotating: 100%|██████████| 602/602 [00:09<00:00, 63.79it/s] 


,doc,text,label
0,doc:1,"[PUTUSAN, Nomor, 48/Pid.B/2024/PN, Bkl, DEMI, ...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
1,doc:2,"[PUTUSAN, Nomor, 42/Pid.B/2024/PN, Bkl, DEMI, ...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
2,doc:3,"[PUTUSAN, Nomor, 49/Pid.B/2024/PN, Bkl, DEMI, ...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
3,doc:4,"[PUTUSAN, Nomor, 56/Pid.B/2024/PN, Bkl, DEMI, ...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
4,doc:5,"[PUTUSAN, Nomor, 57/Pid.B/2024/PN, Bkl, DEMI, ...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
...,...,...,...
597,doc:598,"[PUTUSAN, Nomor, 274/Pid.Sus/2021/PN, Smp, DEM...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
598,doc:599,"[PUTUSAN, Nomor, 272/Pid.B/2021/PN, Smp, DEMI,...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
599,doc:600,"[PUTUSAN, Nomor, 220/Pid.Sus/2021/PN, Smp, DEM...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."
600,doc:601,"[PUTUSAN, Nomor, 293/Pid.B/2021/PN, Smp, DEMI,...","[O, O, B_VERN, I_VERN, O, O, O, O, O, O, O, O,..."


In [64]:
accuracy(count_label)

Dokumen Ke-0 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-1 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-2 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-3 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-4 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-5 => Data Terlebel : 10 ; Total Label : 11 ; Label Hilang : {'B_DEFN'}
Dokumen Ke-6 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-7 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-8 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-9 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-10 => Data Terlebel : 11 ; Total Label : 11 ; Label Hilang : set()
Dokumen Ke-11 => Data Terlebel : 10 ; Total Label : 11 ; Label Hilang : {'B_REGI'}
Dokumen Ke-12 => Data Terlebel : 11 ; Total Label : 11 ; Label H

'Accuracy : 93.6241610738255'

In [65]:
labels[0]

{'dewi ika agustina': 'B_PROS',
 'tislan bin mukamar': 'B_DEFN',
 'wahyu eko suryowati': 'B_JUDG',
 'satrio budiono': 'B_JUDG',
 '1 (satu) tahun dan 10 (sepuluh) bulan': 'B_PUNI',
 '2 (dua) tahun 2 (dua) bulan': 'B_PENA',
 '48/pid.b/2024/pn bkl': 'B_VERN',
 '2 mei 2024': 'B_TIMV',
 'ernila widikartikawati': 'B_JUDP',
 'hosnol bakri': 'B_REGI',
 'pencurian dalam keadaan memberatkan': 'B_CRIA',
 'pasal 363 ayat (2) kuhp': 'B_ARTV'}

## tokenization

In [66]:
def split_token(dataset):
    dataset_copy = dataset.copy()  # Salin dataset agar tidak memodifikasi data asli
    for i, row in tqdm(dataset_copy.iterrows(), total=len(dataset_copy), desc="Processing rows"):
        for index, (text, tag) in enumerate(zip(row["text"], row["label"])):
            if tag != "O":
                token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
                add_label = []
                for i in range(len(token)):
                    if 'B' in tag and not add_label:
                        add_label.append(tag)
                    else:
                        if token[i] != ';':
                            add_label.append(tag.replace('B', 'I'))
                        else:
                            add_label.append('O')

                del row["text"][index]
                del row["label"][index]
                row["text"][index:index] = token
                row["label"][index:index] = add_label
            else:
                token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
                add_label = ['O'] * len(token)  # Set label 'O' untuk semua token

                del row["text"][index]
                del row["label"][index]
                row["text"][index:index] = token
                row["label"][index:index] = add_label
    return dataset_copy

splited_df = split_token(new_df)
splited_df

Processing rows: 100%|██████████| 602/602 [00:54<00:00, 11.15it/s]


,doc,text,label
0,doc:1,"[PUTUSAN, Nomor, 48, /, Pid, ., B, /, 2024, /,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
1,doc:2,"[PUTUSAN, Nomor, 42, /, Pid, ., B, /, 2024, /,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
2,doc:3,"[PUTUSAN, Nomor, 49, /, Pid, ., B, /, 2024, /,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
3,doc:4,"[PUTUSAN, Nomor, 56, /, Pid, ., B, /, 2024, /,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
4,doc:5,"[PUTUSAN, Nomor, 57, /, Pid, ., B, /, 2024, /,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...,...
597,doc:598,"[PUTUSAN, Nomor, 274, /, Pid, ., Sus, /, 2021,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
598,doc:599,"[PUTUSAN, Nomor, 272, /, Pid, ., B, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
599,doc:600,"[PUTUSAN, Nomor, 220, /, Pid, ., Sus, /, 2021,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
600,doc:601,"[PUTUSAN, Nomor, 293, /, Pid, ., B, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."


In [67]:
filtered_rows = new_df.loc[new_df['doc'] == f'doc:{1}']
text_example = []
for sentence_tokens in filtered_rows['text']:
    text_example.extend(sentence_tokens)
label_example = []
for sentence_tokens in filtered_rows['label']:
    label_example.extend(sentence_tokens)
print(text_example)
print(label_example)

['PUTUSAN', 'Nomor', '48', '/', 'Pid', '.', 'B', '/', '2024', '/', 'PN', 'Bkl', 'DEMI', 'KEADILAN', 'BERDASARKAN', 'KETUHANAN', 'YANG', 'MAHA', 'ESA', 'Pengadilan', 'Negeri', 'Bangkalan', 'yang', 'mengadili', 'perkara', 'pidana', 'dengan', 'acara', 'pemeriksaan', 'biasa', 'dalam', 'tingkat', 'pertama', 'menjatuhkan', 'putusan', 'sebagai', 'berikut', 'dalam', 'perkara', 'Terdakwa', ':', '1', '.', 'Nama', 'lengkap', ':', 'Tislan', 'Bin', 'Mukamar', '2', '.', 'Tempat', 'lahir', ':', 'Bangkalan', '3', '.', 'Umur', '/', 'Tanggal', 'lahir', ':', '35', 'Tahun', '/', '12', 'April', '1989', '4', '.', 'Jenis', 'kelamin', ':', 'Laki-laki', '5', '.', 'Kebangsaan', ':', 'Indonesia', '6', '.', 'Tempat', 'tinggal', ':', 'Dsn', '.', 'Lang', 'Lebar', 'Ds', '.', 'Batobella', 'Kec', '.', 'Geger', 'Kab', '.', 'Bangkalan', '7', '.', 'Agama', ':', 'Islam', '8', '.', 'Pekerjaan', ':', 'Wiraswasta', 'Terdakwa', 'Tislan', 'Bin', 'Mukamar', 'ditahan', 'dalam', 'tahanan', 'oleh', ':', '1', '.', 'Penyidik', 'seja

In [68]:
for token, label in zip(text_example, label_example):
  if label != 'O' or token == ' ':
    print(f"Token Text : {token} => {label}")

Token Text : 48 => B_VERN
Token Text : / => I_VERN
Token Text : Pid => I_VERN
Token Text : . => I_VERN
Token Text : B => I_VERN
Token Text : / => I_VERN
Token Text : 2024 => I_VERN
Token Text : / => I_VERN
Token Text : PN => I_VERN
Token Text : Bkl => I_VERN
Token Text : Tislan => B_DEFN
Token Text : Bin => I_DEFN
Token Text : Mukamar => I_DEFN
Token Text : Tislan => B_DEFN
Token Text : Bin => I_DEFN
Token Text : Mukamar => I_DEFN
Token Text : 48 => B_VERN
Token Text : / => I_VERN
Token Text : Pid => I_VERN
Token Text : . => I_VERN
Token Text : B => I_VERN
Token Text : / => I_VERN
Token Text : 2024 => I_VERN
Token Text : / => I_VERN
Token Text : PN => I_VERN
Token Text : Bkl => I_VERN
Token Text : 48 => B_VERN
Token Text : / => I_VERN
Token Text : Pid => I_VERN
Token Text : . => I_VERN
Token Text : B => I_VERN
Token Text : / => I_VERN
Token Text : 2024 => I_VERN
Token Text : / => I_VERN
Token Text : PN => I_VERN
Token Text : Bkl => I_VERN
Token Text : TISLAN => B_DEFN
Token Text : Bin 

## ML representation

In [69]:
import pandas as pd

new_rows = []
fragment_id = 1

for _, row in splited_df.iterrows():
    fragments, labels, fragment, label = [], [], [], []

    for word, label_word in zip(row['text'], row['label']):
        if word != ';':
            fragment.append(word)
            label.append(label_word)
        else:
            if fragment:
                fragments.append(fragment)
                labels.append(label)
            fragment, label = [], []

    if fragment:
        fragments.append(fragment)
        labels.append(label)

    for i, (fragment, label) in enumerate(zip(fragments, labels), start=1):
        new_rows.append({'text': fragment, 'label': label, 'doc': row['doc'], 'fragment': f'fragment:{fragment_id}'})
        fragment_id += 1

df_perfrag = pd.DataFrame(new_rows)

df_pertoken = pd.DataFrame(
    [{'text': token, 'label': label, 'doc': row['doc'], 'fragment': row['fragment']}
     for _, row in df_perfrag.iterrows()
     for token, label in zip(row['text'], row['label'])]
)

df_pertoken = df_pertoken.rename(columns={'text': 'token'}).reindex(columns=['doc','fragment', 'token', 'label'])
df_pertoken

,doc,fragment,token,label
0,doc:1,fragment:1,PUTUSAN,O
1,doc:1,fragment:1,Nomor,O
2,doc:1,fragment:1,48,B_VERN
3,doc:1,fragment:1,/,I_VERN
4,doc:1,fragment:1,Pid,I_VERN
...,...,...,...,...
4094584,doc:602,fragment:78485,Panitera,O
4094585,doc:602,fragment:78485,Pengganti,O
4094586,doc:602,fragment:78485,",",O
4094587,doc:602,fragment:78485,ttd,O


In [70]:
PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
df_pertoken.to_csv(f"{PATH}/ml_format_nopos.csv", index=False)

## POS Tagging

In [77]:
_= !pip install flair
_= !pip install tqdm
!pip install "flair[word-embeddings]"

In [80]:
import flair.datasets
from flair.embeddings import TokenEmbeddings, WordEmbeddings, StackedEmbeddings
from typing import List
from flair.data import Sentence
from flair.models import SequenceTagger
import re
import pandas as pd
from tqdm import tqdm
import shutil
from flair.trainers import ModelTrainer

### training pos tagging model

In [81]:
#load data bahasa indonesia
corpus = flair.datasets.UD_INDONESIAN()

tag_type = 'upos'
tag_dictionary = corpus.make_tag_dictionary(tag_type=tag_type)

#mengubah ke embedding
embedding_types: List[TokenEmbeddings] = [
 WordEmbeddings('id-crawl'),
 WordEmbeddings('id'),
]
embeddings: StackedEmbeddings = StackedEmbeddings(embeddings=embedding_types)

tagger: SequenceTagger = SequenceTagger(hidden_size=256,
                                        embeddings=embeddings,
                                        tag_dictionary=tag_dictionary,
                                        tag_type=tag_type,
                                        use_crf=True)

#training model pos
trainer: ModelTrainer = ModelTrainer(tagger, corpus)
trainer.train('resources/taggers/example-universal-pos',
              learning_rate=0.1,
              mini_batch_size=32,
              max_epochs=10)

# Menyalin model ke Google Drive
model_path = 'resources/taggers/example-universal-pos/best-model.pt'
PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
drive_model_path = f"{PATH}/best-model.pt"
shutil.copy(model_path, drive_model_path)

2026-10-09 08:30:03,023 Reading data from /root/.flair/datasets/ud_indonesian
2026-10-09 08:30:03,029 Train: /root/.flair/datasets/ud_indonesian/id_gsd-ud-train.conllu
2026-10-09 08:30:03,031 Dev: /root/.flair/datasets/ud_indonesian/id_gsd-ud-dev.conllu
2026-10-09 08:30:03,033 Test: /root/.flair/datasets/ud_indonesian/id_gsd-ud-test.conllu


/tmp/ipykernel_696/2919567095.py:5: DeprecationWarning: Call to deprecated method make_tag_dictionary. (Use 'make_label_dictionary' instead.) -- Deprecated since version 0.8.
  tag_dictionary = corpus.make_tag_dictionary(tag_type=tag_type)


2026-10-09 08:30:27,648 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>
2026-10-09 08:30:27,834 ----------------------------------------------------------------------------------------------------
2026-10-09 08:30:27,835 Model: "SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      'id-crawl'
      (embedding): Embedding(1000000, 300)
    )
    (list_embedding_1): WordEmbeddings(
      'id'
      (embedding): Embedding(300686, 300)
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=600, out_features=600, bias=True)
  (rnn): LSTM(600, 256, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=512, out_features=20, bias=True)
  (loss_function): ViterbiLoss()
  (crf): CRF()
)"
2026-10-09 08:30:27,836 -------------------------------------------

/usr/local/lib/python3.13/dist-packages/flair/trainers/trainer.py:545: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


2026-10-09 08:30:32,207 epoch 1 - iter 14/141 - loss 2.53190135 - time (sec): 4.35 - samples/sec: 2283.00 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:34,853 epoch 1 - iter 28/141 - loss 2.29689477 - time (sec): 6.99 - samples/sec: 2812.00 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:36,549 epoch 1 - iter 42/141 - loss 2.09255460 - time (sec): 8.69 - samples/sec: 3420.33 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:38,221 epoch 1 - iter 56/141 - loss 1.93826436 - time (sec): 10.36 - samples/sec: 3826.47 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:40,024 epoch 1 - iter 70/141 - loss 1.81577902 - time (sec): 12.16 - samples/sec: 4057.06 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:41,976 epoch 1 - iter 84/141 - loss 1.70451714 - time (sec): 14.12 - samples/sec: 4181.39 - lr: 0.100000 - momentum: 0.000000
2026-10-09 08:30:43,726 epoch 1 - iter 98/141 - loss 1.60864630 - time (sec): 15.87 - samples/sec: 4332.07 - lr: 0.100000 - momentum: 0.000000
20

...

100%|██████████| 9/9 [00:02<00:00,  3.49it/s]


2026-10-09 08:37:11,238 DEV : loss 0.23011402785778046 - f1-score (micro avg)  0.9218
2026-10-09 08:37:11,267  - 0 epochs without improvement
2026-10-09 08:37:11,268 saving best model
2026-10-09 08:38:12,263 ----------------------------------------------------------------------------------------------------
2026-10-09 08:38:12,267 Loading model from best epoch ...
2026-10-09 08:38:23,527 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>


100%|██████████| 9/9 [00:01<00:00,  5.69it/s]

2026-10-09 08:38:26,027 
Results:
- F-score (micro) 0.9259
- F-score (macro) 0.8829
- Accuracy 0.9259

By class:
              precision    recall  f1-score   support

        NOUN     0.8715    0.9191    0.8947      2509
       PROPN     0.9272    0.9135    0.9203      2162
       PUNCT     0.9932    0.9988    0.9960      1614
        VERB     0.9282    0.9467    0.9373      1256
         ADP     0.9428    0.9470    0.9449      1114
        PRON     0.9419    0.9814    0.9612       644
         ADJ     0.8973    0.6817    0.7748       487
         NUM     0.9764    0.9662    0.9713       385
       CCONJ     0.9808    0.9835    0.9822       364
         ADV     0.8497    0.8123    0.8306       341
         DET     0.9736    0.8728    0.9204       338
         AUX     0.9540    0.9956    0.9744       229
       SCONJ     0.8135    0.8093    0.8114       194
        PART     0.9053    0.9663    0.9348        89
           X     0.6667    0.2500    0.3636        24
         SYM     1.000

'/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data/best-model.pt'

### pos tagging data

nb:

untuk proses pos tagging sebaiknya dijalankan pada colab yang berbeda, karena memakan waktu lama

selain itu jika data terlalu banyak, data dibagi menjadi beberapa bagian sebelum masuk pos tagging, karena jika data yang diproses terlalu banyak maka runtime colab akan putus


In [84]:
import torch
import flair

# 1. Bersihkan sisa memori jika ada proses yang menggantung
torch.cuda.empty_cache()

# 2. Cek apakah Google Colab sudah mendeteksi GPU
gpu_tersedia = torch.cuda.is_available()
print(f"Status GPU: {'TERSEDIA (Bisa digunakan)' if gpu_tersedia else 'TIDAK TERSEDIA (Berjalan di CPU)'}")

if gpu_tersedia:
    nama_gpu = torch.cuda.get_device_name(0)
    print(f"Nama Hardware GPU: {nama_gpu}")

    # 3. CARA MENGGUNAKANNYA: Paksa Flair menggunakan GPU secara global
    flair.device = torch.device('cuda:0')
    print("-> SUKSES: Sistem Flair telah dialihkan ke GPU!")
else:
    print("-> PERINGATAN: Periksa kembali menu 'Runtime > Change runtime type' dan pastikan pilih T4 GPU.")


Status GPU: TERSEDIA (Bisa digunakan)
Nama Hardware GPU: Tesla T4
-> SUKSES: Sistem Flair telah dialihkan ke GPU!


In [82]:
# PATH = '/content/drive/MyDrive/Skripsi/Code/Data'
# pos1 = pd.read_csv(f"{PATH}/pos_data1.csv")
# pos2 = pd.read_csv(f"{PATH}/pos_data2.csv")
# pos = pd.concat([pos1, pos2], ignore_index=True)
# pos

In [ ]:
# # PATH = '/content/drive/MyDrive/Skripsi/Code/Data'
# PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'

# df = pd.read_csv(f"{PATH}/ml_format_nopos.csv")
# token = df['token'].tolist()
# # token1 = token[:len(token)//2]
# # token2 = token[len(token)//2:]

# #tag semua token data legal
# # model_path = '/content/drive/MyDrive/Skripsi/Code'
# model_path = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'

# tag_pos = SequenceTagger.load(f"{model_path}/best-model.pt")
# f_pos = []
# ## for every sentence ##
# for i in tqdm(token):
#   sentence = Sentence(str(i))
#   tag_pos.predict(sentence)
#   ## append tagged sentence ##
#   f_pos.append(sentence.to_tagged_string())

In [3]:
from flair.data import Sentence
from flair.models import SequenceTagger
from tqdm import tqdm
import pandas as pd
import os

PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
df = pd.read_csv(f"{PATH}/ml_format_nopos.csv")
token = df['token'].tolist()

# 1. Load model POS Tagger
tag_pos = SequenceTagger.load(f"{PATH}/best-model.pt")

# File untuk menyimpan hasil sementara (checkpoint)
output_file = f"{PATH}/ml_format_hasil_sementara.csv"
batch_size = 64

print("Memulai prediksi POS Tagging dengan GPU & Checkpoint...")

# Buka file untuk menulis (mode 'a' = append / tambah ke baris bawahnya)
# Jika file belum ada, kita buat headernya dulu
if not os.path.exists(output_file):
    pd.DataFrame(columns=["tagged_text"]).to_csv(output_file, index=False)

# Proses per batch
for i in tqdm(range(0, len(token), batch_size)):
    batch_texts = token[i:i+batch_size]

    # Buat objek sentence
    sentences = [Sentence(str(text)) for text in batch_texts]

    # Prediksi
    tag_pos.predict(sentences)

    # Ambil hasil teks
    batch_results = [s.to_tagged_string() for s in sentences]

    # Simpan langsung ke Google Drive per batch
    pd.DataFrame({"tagged_text": batch_results}).to_csv(output_file, mode='a', header=False, index=False)

print(f"Proses selesai! Hasil aman tersimpan di: {output_file}")

2026-10-09 09:55:15,712 SequenceTagger predicts: Dictionary with 20 tags: O, PROPN, AUX, DET, NOUN, PRON, VERB, ADP, PUNCT, ADV, CCONJ, SCONJ, NUM, ADJ, PART, SYM, INTJ, X, <START>, <STOP>
Memulai prediksi POS Tagging dengan GPU & Checkpoint...


100%|██████████| 63978/63978 [18:55<00:00, 56.36it/s]

Proses selesai! Hasil aman tersimpan di: /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data/ml_format_hasil_sementara.csv


In [5]:
PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'

# 1. Baca file hasil sementara dari Google Drive
df_hasil = pd.read_csv(f"{PATH}/ml_format_hasil_sementara.csv")

# 2. Ambil kolom hasil tagging sebagai list 'f_pos'
f_pos = df_hasil['tagged_text'].astype(str).tolist()

pos = []
for i in tqdm(f_pos):
  m = re.split(r'/', i)
  pos.append(m[-1][:-1])

df['pos'] = pos
# posdf = {"pos": pos}
# posdf = pd.DataFrame(posdf)
# posdf.to_csv(f"{PATH}/pos_data1.csv", index=False)

100%|██████████| 4094589/4094589 [00:06<00:00, 602698.08it/s]


**proses feature engineering sampai evaluation pindah ke kaggle**

## feature engineering

nb:

untuk bagian feature engineering dan modelling sebaiknya dijalankan kaggle, karena colab free tidak mensupport komputasi yang besar

kode feature engineering dan modelling terletak pada beda file

In [6]:
# PATH2 = '/content/drive/MyDrive/Skripsi/Code/Data'
PATH2 = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'

df = pd.read_csv(f"{PATH2}/data_mlformat_clean.csv")
df = df.fillna(method='ffill')
df

/tmp/ipykernel_24907/2731625740.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


,doc,fragment,token,label,pos
0,doc:1,fragment:1,PUTUSAN,O,PROPN
1,doc:1,fragment:1,Nomor,O,PROPN
2,doc:1,fragment:1,48,B_VERN,NUM
3,doc:1,fragment:1,/,I_VERN,PUNCT
4,doc:1,fragment:1,Pid,I_VERN,PROPN
...,...,...,...,...,...
1252323,doc:602,fragment:11301,Panitera,O,PROPN
1252324,doc:602,fragment:11301,Pengganti,O,PROPN
1252325,doc:602,fragment:11301,",",O,PUNCT
1252326,doc:602,fragment:11301,ttd,O,PROPN


In [7]:
class FragmentGetter(object):

    def __init__(self, data):
        self.n_frag = 1
        self.data = data
        self.empty = False
        agg_func = lambda f: [(t, p, l) for t, p, l in
                              zip(f['token'].values.tolist(),
                                  f['pos'].values.tolist(),
                                  f['label'].values.tolist())]
        self.grouped = self.data.groupby('fragment').apply(agg_func)
        self.fragments = [f for f in self.grouped]

    def get_next(self):
        try:
            f = self.grouped['Fragment: {}'.format(self.n_frag)]
            self.n_frag += 1
            return f
        except:
            return None

In [8]:
def token2features(frag, i):
    token = frag[i][0]
    postag = frag[i][1]

    features = {
        'bias': 1.0,
        'token.lower()': token.lower(),
        'token.isupper()': token.isupper(),
        'token.istitle()': token.istitle(),
        'token.isdigit()': token.isdigit(),
        'postag': postag,
    }

    # Features for previous token
    if i > 0:
        features.update({
            'prev1.lower()': frag[i-1][0].lower(),
            'prev1.postag': frag[i-1][1],
        })
    else:
        features['BOF'] = True  # Beginning of fragment

    if i > 1:
        features.update({
            'prev2.lower()': frag[i-2][0].lower(),
            'prev2.postag': frag[i-2][1],
        })

    # Features for next token
    if i < len(frag) - 1:
        features.update({
            'next1.lower()': frag[i+1][0].lower(),
            'next1.postag': frag[i+1][1],
        })
    else:
        features['EOF'] = True  # End of fragment

    if i < len(frag) - 2:
        features.update({
            'next2.lower()': frag[i+2][0].lower(),
            'next2.postag': frag[i+2][1],
        })

    return features

def frag2features(frag):
  return [token2features(frag, i) for i in range(len(frag))]
def frag2labels(frag):
  return [label for token, postag, label in frag]

In [9]:
X_crf = df.drop(['doc'], axis=1)
getter = FragmentGetter(X_crf)
fragments = getter.fragments

X = [frag2features(f) for f in fragments]
y = [frag2labels(f) for f in fragments]

/tmp/ipykernel_24907/3413044544.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby('fragment').apply(agg_func)


# Modelling

In [10]:
_= !pip install sklearn-crfsuite
_= !pip install jcopml
_= !pip install eli5

In [11]:
import sklearn
from sklearn.model_selection import train_test_split
import sklearn_crfsuite
from sklearn_crfsuite import CRF
from sklearn_crfsuite.utils import flatten
from sklearn_crfsuite import scorers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, multilabel_confusion_matrix

In [12]:
import pandas as pd
import numpy as np

# Mengumpulkan data per dokumen
docs = df['doc'].unique().tolist()

# Mengacak dokumen
np.random.seed(42)  # Untuk reproduktibilitas, bisa diubah sesuai kebutuhan
np.random.shuffle(docs)

# Menentukan batas dokumen terakhir untuk training (70% dari total dokumen)
train_doc_count = int(len(docs) * 0.7)
train_docs = docs[:train_doc_count]

# Membuat DataFrame untuk data train
train_df = df[df['doc'].isin(train_docs)]

# Mencari fragment terakhir dari data train
id_last_frag = train_df['fragment'].unique().tolist()[-1]
last_frag = int(id_last_frag.split(':')[1])

# Memotong data menjadi training dan testing set berdasarkan indeks terakhir
X_train = X[:last_frag+1]
y_train = y[:last_frag+1]
X_test = X[last_frag+1:]
y_test = y[last_frag+1:]

# Mengecek hasil
print(f'Total dokumen: {len(docs)}')
print(f'Total dokumen training: {train_doc_count}')
print(f'Index fragment terakhir untuk training: {last_frag}')

Total dokumen: 602
Total dokumen training: 421
Index fragment terakhir untuk training: 11301


In [13]:
import time

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf_start = time.time()
crf.fit(X_train, y_train)
crf_end = time.time()
print(f'{(crf_end - crf_start):.2f} Detik.')

482.76 Detik.


In [36]:
import joblib
import os

PATH = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
os.makedirs(PATH, exist_ok=True)
model_path = f'{PATH2}/600crf.pkl'
joblib.dump(crf, model_path)
print(f'Model Tersimpan di {model_path}')

crf_check = joblib.load(model_path)
print(f'Verifikasin load OK', type(crf_check))

Model Tersimpan di /content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data/600crf.pkl
Verifikasin load OK <class 'sklearn_crfsuite.estimator.CRF'>


# Evaluation

In [37]:
!pip install matplotlib
!pip install seaborn

In [38]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
y_pred= crf.predict(X_test)
y_test_flat = [label for sent in y_test for label in sent]
y_pred_flat = [label for sent in y_pred for label in sent]

# change label to list
y = df.label.values
classes = np.unique(y)
classes = classes.tolist()

# Buat confusion matrix
cm = confusion_matrix(y_test_flat, y_pred_flat, labels=classes)

# Visualisasikan confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_test_flat, y_pred_flat, labels=classes))

In [ ]:
_= !pip install eli5

# Testing

In [ ]:
_= !pip install PyPDF2
_= !pip install tqdm
_= !pip install jcopml
_= !pip install fuzzywuzzy python-Levenshtein
_= !pip install sklearn-crfsuite

In [ ]:
import requests
import PyPDF2
import re
import pandas as pd
import zipfile
import os
from tqdm import tqdm
import joblib

## Read PDF

In [ ]:
def clean_page(text):
    text = text.replace("Mahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nDirektori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n", "")
    text = text.replace("\nDisclaimer\nKepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\npelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\nDalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\nEmail : kepaniteraan@mahkamahagung.go.id", "")
    text = text.replace("Telp : 021-384 3348 (ext.318)", "")
    text = re.sub(r'\nHalaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'Halaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'\nHal. \d+ dari \d+ .*', '', text)
    text = re.sub(r'Hal. \d+ dari \d+ .*', '', text)
    return text.strip()

def read_pdf(file_pdf):
  try:
    pdf_text = ''
    pdf_file = open(file_pdf, 'rb')
    pdf_reader = PyPDF2.PdfReader(pdf_file)

    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        text = clean_page(page.extract_text())
        # text = page.extract_text()

        pdf_text += ' ' + text

    pdf_file.close()
    return pdf_text.strip()

  except requests.exceptions.RequestException as e:
    print("Error:", e)

## Preprocessing

In [ ]:
def clean_text(text):
    text = text.replace('P U T U S A N', 'PUTUSAN').replace('T erdakwa', 'Terdakwa').replace('T empat', 'Tempat').replace('T ahun', 'Tahun')
    text = text.replace('P  E  N  E  T  A  P  A  N', 'PENETAPAN').replace('J u m l a h', 'Jumlah').replace('M E N G A D I L I', 'MENGADILI')
    text = re.sub(r'Halaman \d+', ' ', text)
    text = text.replace('\uf0d8', '').replace('\uf0b7', '').replace('\n', ' ')
    text = re.sub(r'([“”"])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

## Representasi CRF

In [ ]:
def representation_crf(text, id_fragment):
    content_results = {"doc": [], "fragment": [], "token": []}

    fragments = text.split(';')

    for fragment_idx, fragment in enumerate(fragments):
        fragment_str = f"fragment:{id_fragment}"
        doc_str = f"doc:1"

        fragment = re.sub(r'([\/,\.():;])', r' \1 ', fragment)
        tokens = re.findall(r'\S+', fragment)

        for token in tokens:
            content_results["doc"].append(doc_str)
            content_results["fragment"].append(fragment_str)
            content_results["token"].append(token)

    new_data = pd.DataFrame(content_results)

    return new_data

## Testing

In [ ]:
class FragmentGetter(object):

    def __init__(self, data):
        self.n_frag = 1
        self.data = data
        self.empty = False
        agg_func = lambda s: [(a) for a in zip(s['token'].values.tolist(),
        # agg_func = lambda s: [(a, b) for a, b in zip(s['token'].values.tolist(),
                                                    # s['pos'].values.tolist(),
                                                     )]
        self.grouped = self.data.groupby('fragment').apply(agg_func)
        self.fragments = [f for f in self.grouped]

    def get_next(self):
        try:
            f = self.grouped['Fragment: {}'.format(self.n_frag)]
            self.n_frag += 1
            return f
        except:
            return None

In [ ]:
def token2features(frag, i):
    token = frag[i][0]
    # postag = frag[i][1]

    features = {
        'bias': 1.0,
        'token':token,
        # 'token.lower()': token.lower(),
        # 'token.isupper()': token.isupper(),
        # 'token.istitle()': token.istitle(),
        # 'token.isdigit()': token.isdigit(),
        # 'postag': postag,
    }

    # Features for previous token
    if i > 0:
        features.update({
            'prev1': frag[i-1][0],
            # 'prev1.lower()': frag[i-1][0].lower(),
            # 'prev1.isupper()': frag[i-1][0].isupper(),
            # 'prev1.istitle()': frag[i-1][0].istitle(),
            # 'prev1.isdigit()': frag[i-1][0].isdigit(),
            # 'prev1.postag': frag[i-1][1],
        })
    else:
        features['BOF'] = True  # Beginning of fragment

    if i > 1:
        features.update({
            'prev2': frag[i-2][0],
            # 'prev2.lower()': frag[i-2][0].lower(),
            # 'prev2.isupper()': frag[i-2][0].isupper(),
            # 'prev2.istitle()': frag[i-2][0].istitle(),
            # 'prev2.isdigit()': frag[i-2][0].isdigit(),
            # 'prev2.postag': frag[i-2][1],
        })

    # Features for next token
    if i < len(frag) - 1:
        features.update({
            'next1': frag[i+1][0],
            # 'next1.lower()': frag[i+1][0].lower(),
            # 'next1.isupper()': frag[i+1][0].isupper(),
            # 'next1.istitle()': frag[i+1][0].istitle(),
            # 'next1.isdigit()': frag[i+1][0].isdigit(),
            # 'next1.postag': frag[i+1][1],
        })
    else:
        features['EOF'] = True  # End of fragment

    if i < len(frag) - 2:
        features.update({
            'next2': frag[i+2][0],
            # 'next2.lower()': frag[i+2][0].lower(),
            # 'next2.isupper()': frag[i+2][0].isupper(),
            # 'next2.istitle()': frag[i+2][0].istitle(),
            # 'next2.isdigit()': frag[i+2][0].isdigit(),
            # 'next2.postag': frag[i+2][1],
        })

    return features

def frag2features(frag):
  return [token2features(frag, i) for i in range(len(frag))]
def frag2labels(frag):
  return [label for token, label in frag]
  # return [label for token, postag, label in frag]

In [ ]:
def check_entity(df, doc_idx):
    filtered_rows = df.loc[df['doc'] == f'doc:{doc_idx}']

    tokens = filtered_rows['token'].tolist()
    labels = filtered_rows['label'].tolist()

    entities = {}
    current_entity = ''
    current_label = ''

    for token, label in zip(tokens, labels):
        if label == 'O':
            continue

        entity_type = label.split('_')[-1]

        if label.startswith('B_'):
            if current_label:
                if current_label not in entities:
                    entities[current_label] = []
                entities[current_label].append(current_entity.strip())

            current_entity = token
            current_label = entity_type

        elif label.startswith('I_') and current_label == entity_type:
            current_entity += ' ' + token

    # Menambahkan entitas terakhir ke dalam dictionary
    if current_label:
        if current_label not in entities:
            entities[current_label] = []
        entities[current_label].append(current_entity.strip())

    for entity_type, entity_list in entities.items():
        for entity in entity_list:
            print(f"{entity_type} = {entity}")

## Main test

In [12]:
from fuzzywuzzy import fuzz
# ------------------------------------------------- read file
pdf_path = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/1_Pid.C_2024_PN_Smp.pdf'
content = read_pdf(pdf_path)
content = clean_text(content)

# Define the paths
PATH2 = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
# model_path = f'{PATH2}/600crf.pkl'
model_path = f'{PATH2}/600crf_prevnext.pkl'

# Split content into sentences
pecahan = (content).split(';')

# Function to process a single sentence and return the resulting data
def process_sentence(sentence, id_fragment):
    data = representation_crf(sentence)

    # Feature engineering
    X_crf = data.drop(['doc'], axis=1)
    getter = FragmentGetter(X_crf)
    fragments = getter.fragments
    X = [frag2features(f) for f in fragments]

    # Predict
    crf = joblib.load(model_path)
    y_pred = crf.predict(X)

    # Input label to dataset
    flat_predictions = [tag for sentence_tags in y_pred for tag in sentence_tags]
    assert len(flat_predictions) == len(data['token'])
    data['label'] = flat_predictions

    return data

# Function to check entities and collect them
def check_entity(df):
    tokens = df['token'].tolist()
    labels = df['label'].tolist()

    entities = {}
    current_entity = ''
    current_label = ''

    for token, label in zip(tokens, labels):
        if label == 'O':
            continue

        entity_type = label.split('_')[-1]

        if label.startswith('B_'):
            if current_label:
                if current_label not in entities:
                    entities[current_label] = set()
                entities[current_label].add(current_entity.strip())

            current_entity = token
            current_label = entity_type

        elif label.startswith('I_') and current_label == entity_type:
            current_entity += ' ' + token

    # Menambahkan entitas terakhir ke dalam dictionary
    if current_label:
        if current_label not in entities:
            entities[current_label] = set()
        entities[current_label].add(current_entity.strip())

    return entities

# Collect results from each sentence
results = []
all_entities = {}
print(len(pecahan))

count = 1
for kalimat in pecahan:
    result = process_sentence(kalimat, count)
    results.append(result)
    count += 1

    # Extract and collect entities
    entities = check_entity(result)
    for entity_type, entity_set in entities.items():
        if entity_type not in all_entities:
            all_entities[entity_type] = set()
        all_entities[entity_type].update(entity_set)

    # Combine all results into a single DataFrame
df = pd.concat(results, ignore_index=True)

# Convert the entities dictionary to a list of tuples and remove duplicates
final_entities = {entity_type: list(entity_set) for entity_type, entity_set in all_entities.items()}

key_mapping = {
    "VERN": "Nomor Putusan",
    "DEFN": "Terdakwa",
    "CRIA": "Tindak pidana",
    "PENA": "Tuntutan Hukuman",
    "ARTV": "Pasal yang Dilanggar",
    "PUNI": "Putusan Hukuman",
    "JUDP": "Hakim Ketua",
    "JUDG": "Hakim Anggota",
    "TIMV": "Tanggal Perkara",
    "REGI": "Panitera",
    "PROS": "Penuntut Umum"
}

def remove_similar(items, threshold=80):
    unique_items = []
    for item in items:
        is_covered = False
        for other_item in unique_items:
            if fuzz.partial_ratio(item, other_item) > threshold:
                is_covered = True
                # Keep the longer item if similar
                if len(item) > len(other_item):
                    unique_items.remove(other_item)
                    unique_items.append(item)
                break
        if not is_covered:
            unique_items.append(item)
    return unique_items

for entity in final_entities:
    final_entities[entity] = remove_similar(final_entities[entity])

for key, value in final_entities.items():
    deskripsi = key_mapping.get(key, key)
    print(f"{deskripsi}: {' | '.join(value)}")

28


/tmp/ipykernel_191/222054561.py:80: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby('fragment').apply(agg_func)
/tmp/ipykernel_191/222054561.py:80: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby('fragment').apply(agg_func)
/tmp/ipykernel_191/222054561.py:80: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavi

In [13]:
def remove_rows_with_all_O(df):
    df_filtered = df[~df.groupby(['doc', 'fragment'])['label'].transform(lambda x: all(label == 'O' for label in x))]
    df_filtered = df_filtered.reset_index(drop=True)
    return df_filtered

# Menghapus data yang memenuhi kondisi
filtered_data = remove_rows_with_all_O(df)
filtered_data.tail()

,doc,fragment,token,label


In [14]:
unique_counts = df.nunique()
print(unique_counts)

doc           1
fragment      1
token       460
label         1
dtype: int64


# App

In [15]:
_= !pip install gradio fuzzywuzzy joblib PyPDF2 sklearn-crfsuite

In [16]:
import re
import joblib
import pandas as pd
import PyPDF2
import gradio as gr
from fuzzywuzzy import fuzz

PATH2 = '/content/drive/MyDrive/Kuliah/Semester 7/Ekstraksi Informasi/Data'
model_path = f'{PATH2}/600crf_prevnext.pkl'

# Function to clean the page
def clean_page(text):
    text = text.replace("Mahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nMahkamah Agung Republik Indonesia\nDirektori Putusan Mahkamah Agung Republik Indonesia\nputusan.mahkamahagung.go.id\n", "")
    text = text.replace("\nDisclaimer\nKepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas\npelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.\nDalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui :\nEmail : kepaniteraan@mahkamahagung.go.id", "")
    text = text.replace("Telp : 021-384 3348 (ext.318)", "")
    text = re.sub(r'\nHalaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'Halaman \d+ dari \d+ .*', '', text)
    text = re.sub(r'\nHal. \d+ dari \d+ .*', '', text)
    text = re.sub(r'Hal. \d+ dari \d+ .*', '', text)
    return text.strip()

# Function to read and clean text from PDF
def read_pdf(file_pdf):
    try:
        pdf_text = ''
        pdf_file = open(file_pdf, 'rb')
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = clean_page(page.extract_text())
            pdf_text += ' ' + text

        pdf_file.close()
        return pdf_text.strip()

    except Exception as e:
        print("Error:", e)

# Function to clean the text
def clean_text(text):
    text = text.replace('P U T U S A N', 'PUTUSAN').replace('T erdakwa', 'Terdakwa').replace('T empat', 'Tempat').replace('T ahun', 'Tahun')
    text = text.replace('P  E  N  E  T  A  P  A  N', 'PENETAPAN').replace('J u m l a h', 'Jumlah').replace('M E N G A D I L I', 'MENGADILI')
    text = re.sub(r'Halaman \d+', ' ', text)
    text = text.replace('\uf0d8', '').replace('\uf0b7', '').replace('\n', ' ')
    text = re.sub(r'([“”"])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Function for CRF representation
def representation_crf(text):
    content_results = {"doc": [], "fragment": [], "token": []}

    fragments = text.split(';')

    for fragment_idx, fragment in enumerate(fragments):
        fragment_str = f"fragment:{fragment_idx}"
        doc_str = f"doc:1"

        fragment = re.sub(r'([\/,\.():;])', r' \1 ', fragment)
        tokens = re.findall(r'\S+', fragment)

        for token in tokens:
            content_results["doc"].append(doc_str)
            content_results["fragment"].append(fragment_str)
            content_results["token"].append(token)

    new_data = pd.DataFrame(content_results)

    return new_data

# Fragment getter class
class FragmentGetter(object):

    def __init__(self, data):
        self.n_frag = 1
        self.data = data
        self.empty = False
        agg_func = lambda s: [(a) for (a) in zip(s['token'].values.tolist())]
        self.grouped = self.data.groupby('fragment').apply(agg_func)
        self.fragments = [f for f in self.grouped]

    def get_next(self):
        try:
            f = self.grouped['Fragment: {}'.format(self.n_frag)]
            self.n_frag += 1
            return f
        except:
            return None

# Function to convert token to features
def token2features(frag, i):
    token = frag[i][0]

    features = {
        'bias': 1.0,
        'token': token
    }

    # Features for previous token
    if i > 0:
        features.update({
            'prev1': frag[i - 1][0]

        })
    else:
        features['BOF'] = True  # Beginning of fragment

    if i > 1:
        features.update({
            'prev2': frag[i - 2][0]
        })

    # Features for next token
    if i < len(frag) - 1:
        features.update({
            'next1': frag[i + 1][0]
        })
    else:
        features['EOF'] = True  # End of fragment

    if i < len(frag) - 2:
        features.update({
            'next2': frag[i + 2][0]
        })

    return features

# Function to convert fragment to features
def frag2features(frag):
    return [token2features(frag, i) for i in range(len(frag))]

def frag2labels(frag):
    return [label for token, label in frag]

# Function to process a single sentence and return the resulting data
def process_sentence(sentence):
    data = representation_crf(sentence)

    # Feature engineering
    X_crf = data.drop(['doc'], axis=1)
    getter = FragmentGetter(X_crf)
    fragments = getter.fragments
    X = [frag2features(f) for f in fragments]

    # Predict
    crf = joblib.load(model_path)
    y_pred = crf.predict(X)

    # Input label to dataset
    flat_predictions = [tag for sentence_tags in y_pred for tag in sentence_tags]
    assert len(flat_predictions) == len(data['token'])
    data['label'] = flat_predictions

    return data

# Function to check entities and collect them
def check_entity(df):
    tokens = df['token'].tolist()
    labels = df['label'].tolist()

    entities = {}
    current_entity = ''
    current_label = ''

    for token, label in zip(tokens, labels):
        if label == 'O':
            continue

        entity_type = label.split('_')[-1]

        if label.startswith('B_'):
            if current_label:
                if current_label not in entities:
                    entities[current_label] = set()
                entities[current_label].add(current_entity.strip())

            current_entity = token
            current_label = entity_type

        elif label.startswith('I_') and current_label == entity_type:
            current_entity += ' ' + token

    # Add the last entity to the dictionary
    if current_label:
        if current_label not in entities:
            entities[current_label] = set()
        entities[current_label].add(current_entity.strip())

    return entities


# Function to format the entities for display
def format_entities(entities):
    key_mapping = {
        "VERN": "Nomor Putusan",
        "DEFN": "Terdakwa",
        "CRIA": "Tindak pidana",
        "PENA": "Tuntutan Hukuman",
        "ARTV": "Pasal yang Dilanggar",
        "PUNI": "Putusan Hukuman",
        "JUDP": "Hakim Ketua",
        "JUDG": "Hakim Anggota",
        "TIMV": "Tanggal Perkara",
        "REGI": "Panitera",
        "PROS": "Penuntut Umum"
    }

    def remove_similar(items, threshold=80):
        unique_items = []
        for item in items:
            is_covered = False
            for other_item in unique_items:
                if fuzz.partial_ratio(item, other_item) > threshold:
                    is_covered = True
                    # Keep the longer item if similar
                    if len(item) > len(other_item):
                        unique_items.remove(other_item)
                        unique_items.append(item)
                    break
            if not is_covered:
                unique_items.append(item)
        return unique_items

    for entity in entities:
        entities[entity] = remove_similar(entities[entity])

    formatted_entities = []
    for key, value in entities.items():
        deskripsi = key_mapping.get(key, key)
        formatted_entities.append(f"{deskripsi}: {' | '.join(value)}")
    return '\n'.join(formatted_entities)

# Function to visualize tokens with labels using HighlightedText
def visualize_ner(df):
    tokens = df['token'].tolist()
    labels = df['label'].tolist()

    entities = []
    current_entity = ""
    current_label = ""

    for token, label in zip(tokens, labels):
        if label == "O":
            if current_entity:
                entities.append((current_entity, current_label))
                current_entity = ""
                current_label = ""
            entities.append((token, None))
        else:
            entity_type = label.split('_')[-1]
            if label.startswith('B_'):
                if current_entity:
                    entities.append((current_entity, current_label))
                current_entity = token
                current_label = entity_type
            elif label.startswith('I_') and current_label == entity_type:
                current_entity += " " + token

    if current_entity:
        entities.append((current_entity, current_label))

    return entities

# Function to process the input text
def process_text(input_text):
    pecahan = input_text.split(';')

    results = []
    all_entities = {}
    visualizations = []

    for kalimat in pecahan:
        result = process_sentence(kalimat)
        results.append(result)

        # Extract and collect entities
        entities = check_entity(result)
        for entity_type, entity_set in entities.items():
            if entity_type not in all_entities:
                all_entities[entity_type] = set()
            all_entities[entity_type].update(entity_set)

        # Generate visualization
        visualizations.extend(visualize_ner(result))

    # Combine all results into a single DataFrame
    df = pd.concat(results, ignore_index=True)

    # Convert the entities dictionary to a list of tuples and remove duplicates
    final_entities = {entity_type: list(entity_set) for entity_type, entity_set in all_entities.items()}

    return format_entities(final_entities), visualizations

# Function to handle uploaded PDF
def process_pdf(pdf_file):
    text = read_pdf(pdf_file.name)
    return process_text(clean_text(text))

# Define Gradio interface using Blocks
with gr.Blocks() as iface:
    gr.Markdown("## Entity Extraction")
    gr.Markdown("Enter your text to extract entities or upload a PDF file.")

    with gr.Tab("Input Text"):
        text_input = gr.Textbox(label="Input Text")
        text_button = gr.Button("Extract Entities")
        text_output_entities = gr.Textbox(label="Extracted Entities", lines=10)
        text_output_visualization = gr.HighlightedText(label="Visualization")
        text_button.click(fn=process_text, inputs=text_input, outputs=[text_output_entities, text_output_visualization])

    with gr.Tab("Upload PDF"):
        pdf_input = gr.File(label="Upload PDF")
        pdf_button = gr.Button("Extract Entities")
        pdf_output_entities = gr.Textbox(label="Extracted Entities", lines=10)
        pdf_output_visualization = gr.HighlightedText(label="Visualization")
        pdf_button.click(fn=process_pdf, inputs=pdf_input, outputs=[pdf_output_entities, pdf_output_visualization])

# Launch Gradio app
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38da5112a39a4cbed4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
